# TGN-JointIDS — NF-ToN-IoT-v3 (research notebook)

Original end-to-end notebook used to produce the results reported in the paper, with outputs cleared and local paths replaced by repository-relative ones. For a modular, scriptable version of the same pipeline see `tgn_jointids/` and `scripts/` (README).


In [ ]:
%matplotlib inline


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install torch_geometric -q
!pip install tqdm seaborn -q


In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import (
    GATConv, SAGPooling, global_mean_pool
)
from torch_geometric.loader import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, balanced_accuracy_score
)
from collections import deque, defaultdict, Counter
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time, copy, warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')


In [ ]:
# =============================================================================
# DATA LOADING & PREPROCESSING
# =============================================================================
EXP2_PATH = 'data/raw/NF-ToN-IoT-v3_exp2_common20_rare100.csv'

df = pd.read_csv(EXP2_PATH)
print('Dataset shape:', df.shape)

df['Attack'] = df['Attack'].astype(str).str.strip().str.lower()
df['FLOW_START_MILLISECONDS'] = pd.to_numeric(
    df['FLOW_START_MILLISECONDS'], errors='coerce')
df.dropna(subset=['FLOW_START_MILLISECONDS'], inplace=True)

# Sort chronologically — REQUIRED for strict temporal split
df = df.sort_values('FLOW_START_MILLISECONDS').reset_index(drop=True)
print('\nAttack distribution:')
print(df['Attack'].value_counts())


## Balance all classes to 120,000 (120K) samples

In [ ]:
TARGET_SAMPLES = 120000

balanced_parts = []

# Use the original attack column, assuming it is 'Attack'
for attack_type, group in df.groupby('Attack'):
    
    if len(group) >= TARGET_SAMPLES:
        sampled = group.sample(n=TARGET_SAMPLES, random_state=42)
    else:
        sampled = group.sample(n=TARGET_SAMPLES, replace=True, random_state=42)
    
    sampled = sampled.copy()
    sampled['Attack'] = attack_type  # keep label
    balanced_parts.append(sampled)

# Combine all classes
df_balanced = pd.concat(balanced_parts, ignore_index=True)

# Sort for TGN temporal logic
df_balanced = df_balanced.sort_values('FLOW_START_MILLISECONDS').reset_index(drop=True)

print('--- Balancing Complete ---')
print('New Dataset shape:', df_balanced.shape)

print('\nNew Attack distribution (all should be 120,000):')
print(df_balanced['Attack'].value_counts())

# Overwrite original df
df = df_balanced

In [ ]:
plt.rcParams.update({
    'figure.figsize': (3.5, 2.5), 'figure.dpi': 300,
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'legend.fontsize': 8, 'axes.grid': False,
})


In [ ]:
counts = df['Attack'].value_counts()
fig, ax = plt.subplots(figsize=(4.5, 2.8), dpi=300)
colors = plt.cm.viridis(np.linspace(0, 1, len(counts)))
bars = ax.bar(counts.index, counts.values, color=colors)
ax.set_yscale('log')
ax.set_title('Dataset Class Distribution (log scale)', fontsize=9)
ax.set_xlabel('Attack Type', fontsize=8)
ax.set_ylabel('Samples', fontsize=8)
plt.xticks(rotation=45, ha='right', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('data_distribution.pdf', bbox_inches='tight')
plt.show()

print('=' * 50)
for cls, count in counts.items():
    print(f'  {cls:<20} {count:>10,}')
print(f'  Total Classes: {len(counts)}')
print(f'  Total Samples: {counts.sum():,}')


In [ ]:
# =============================================================================
# FEATURE DEFINITIONS
# =============================================================================
STATISTICAL_FEATURES = [
    'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS',
    'TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS',
    'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT',
    'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PK TS_128_TO_256_BYTES',
    'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES',
    'NUM_PKTS_1024_TO_1514_BYTES'
]

BEHAVIORAL_FEATURES = [
    'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX',
    'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV',
    'DST_TO_SRC_IAT_MIN', 'DST_TO_SRC_IAT_MAX',
    'DST_TO_SRC_IAT_AVG', 'DST_TO_SRC_IAT_STDDEV'
]

NODE_FEATURES = STATISTICAL_FEATURES + BEHAVIORAL_FEATURES   # 21 raw features
EDGE_FEATURES = ['IN_BYTES', 'OUT_BYTES', 'FLOW_DURATION_MILLISECONDS', 'TCP_FLAGS']

# Full ordered class list — keeps mapping stable across runs
ATTACK_CLASSES = [
    'benign', 'ddos', 'xss', 'password', 'scanning',
    'backdoor', 'injection', 'dos', 'mitm', 'ransomware'
]
attack2id = {a: i for i, a in enumerate(ATTACK_CLASSES)}
id2attack  = {i: a for a, i in attack2id.items()}
NUM_CLASSES = len(ATTACK_CLASSES)

def safe_attack_to_id(lbl):
    lbl = str(lbl).lower()
    return attack2id.get(lbl, attack2id['benign'])

print(f'Node raw features  : {len(NODE_FEATURES)}')
print(f'Edge raw features  : {len(EDGE_FEATURES)}')
print(f'Attack classes     : {NUM_CLASSES}')
print(f'attack2id          : {attack2id}')


In [ ]:
# =============================================================================
# TGN NODE MEMORY
# Implements persistent per-node memory vectors (Rossi et al., NeurIPS 2020).
# Memory is keyed by integer IPv4 hash, survives DataLoader batching.
# =============================================================================

def ip_to_int(ip_str):
    """Convert IPv4 string 'a.b.c.d' to a unique positive int64."""
    parts = str(ip_str).split('.')
    try:
        val = 0
        for p in parts:
            val = val * 256 + int(p)
        return val & 0x7FFFFFFFFFFFFFFF
    except Exception:
        return abs(hash(ip_str)) & 0x7FFFFFFFFFFFFFFF


# =============================================================================
# TGN NODE MEMORY (IMPROVED VERSION)
# Adds: temporal decay + message buffering + safer GRU update
# Inspired by Temporal Graph Networks (Rossi et al., 2020)
# =============================================================================

import torch
import torch.nn as nn
import math


def ip_to_int(ip_str):
    """Convert IPv4 string to stable integer ID."""
    try:
        parts = str(ip_str).split('.')
        val = 0
        for p in parts:
            val = val * 256 + int(p)
        return val & 0x7FFFFFFFFFFFFFFF
    except Exception:
        return abs(hash(str(ip_str))) & 0x7FFFFFFFFFFFFFFF


class TGNNodeMemory:
    def __init__(self, memory_dim, decay=0.98):
        self.memory_dim = memory_dim
        self.decay = decay

        self.memory = {}        # node_id -> tensor
        self.last_update = {}   # node_id -> timestamp proxy (step index)
        self._step = 0
        self._n_new = 0

    # -------------------------------------------------------------
    def get(self, node_ids, device):
        mems = []
        for nid in node_ids:
            nid = int(nid)

            if nid not in self.memory:
                self.memory[nid] = torch.zeros(self.memory_dim)
                self.last_update[nid] = self._step
                self._n_new += 1

            # temporal decay (VERY IMPORTANT FIX)
            time_gap = self._step - self.last_update[nid]
            decay_factor = self.decay ** time_gap

            mems.append(self.memory[nid] * decay_factor)

        return torch.stack(mems).to(device)

    # -------------------------------------------------------------
    def update_with_gru(self, node_ids, new_messages, gru_cell, device):
        """
        Improved update:
        - applies temporal awareness
        - stabilizes rare-class learning
        """
        old_mems = self.get(node_ids, device)
        msgs = new_messages.detach().to(device)

        with torch.no_grad():
            updated = gru_cell(msgs, old_mems)

        updated = updated.cpu()

        for i, nid in enumerate(node_ids):
            nid = int(nid)

            # exponential moving update (stability boost)
            if nid in self.memory:
                self.memory[nid] = 0.7 * self.memory[nid] + 0.3 * updated[i]
            else:
                self.memory[nid] = updated[i]

            self.last_update[nid] = self._step

        self._step += 1

    # -------------------------------------------------------------
    def reset(self):
        self.memory.clear()
        self.last_update.clear()
        self._step = 0
        self._n_new = 0

    # -------------------------------------------------------------
    def __len__(self):
        return len(self.memory)

    def stats(self):
        return (
            f"Nodes={len(self.memory)} | "
            f"New={self._n_new} | "
            f"Step={self._step}"
        )


print("Improved TGNNodeMemory defined.")


In [ ]:
# =============================================================================
# ENS CLASS WEIGHTING  (Cui et al., 2019 CVPR)
# Replaces inverse-frequency weighting whose extreme ratios caused instability.
# =============================================================================

def compute_ens_weights(class_counts, beta=0.9999, device='cpu'):
    """
    Effective Number of Samples weighting.
    w_c = (1 - beta) / (1 - beta^n_c),  then normalized so mean == 1.
    beta=0.9999 compresses extreme weight ratios vs inverse-frequency.
    """
    counts = np.array(class_counts, dtype=np.float64)
    counts = np.where(counts <= 0, 1.0, counts)   # avoid log(0)
    eff_num = (1.0 - beta ** counts) / (1.0 - beta)
    w = 1.0 / eff_num
    w = w / (w.mean() + 1e-12)                    # normalize
    return torch.tensor(w, dtype=torch.float32).to(device)


# =============================================================================
# FOCAL LOSS  [FIX 4: replaces cross_entropy on the multi-class head]
#
# WHY focal loss here?
#   Fix 1 (ENS weights) ensures that *ransomware errors count more than DDoS
#   errors* — that is inter-class rebalancing.
#   But the balanced dataset forces 120k ransomware graphs, almost all of which
#   are duplicates of 3,971 original flows. After a few epochs the model
#   already predicts those duplicate patterns with high confidence (easy
#   examples). Cross-entropy accumulates large gradient mass from these easy
#   correct predictions, drowning out the signal from the hard boundary cases
#   the model still gets wrong.
#
#   Focal loss = -alpha_t * (1 - p_t)^gamma * log(p_t)
#   The (1 - p_t)^gamma term shrinks the loss for high-confidence correct
#   predictions and keeps it large for low-confidence (hard) ones.
#   With gamma=2 a prediction at p=0.9 is down-weighted by 0.01x relative
#   to a prediction at p=0.5 — the model is forced to keep learning from
#   the hard ransomware / scanning boundary cases throughout training.
#
#   label_smoothing=0.05 is preserved from the original implementation to
#   prevent overconfident logits, which is especially important when focal
#   loss is used (high gamma would otherwise collapse entropy too fast).
#
#   The binary head keeps plain cross_entropy — it is already well-balanced
#   and the added complexity of focal loss there is not warranted.
# =============================================================================

def focal_loss_with_smoothing(logits, targets, weight=None,
                               gamma=2.0, label_smoothing=0.05):
    """
    Focal loss with label smoothing for the multi-class attack head.
    Args:
        logits          : [B, C] raw model outputs
        targets         : [B]   integer class indices
        weight          : [C]   per-class ENS weights (alpha in focal notation)
        gamma           : focusing parameter; 2.0 is the standard default
        label_smoothing : prevents overconfident logits (keep at 0.05)
    Returns:
        scalar loss
    """
    # Step 1: cross-entropy with label smoothing gives -log(p_t) per sample
    ce_per_sample = F.cross_entropy(
        logits, targets,
        weight=weight,
        label_smoothing=label_smoothing,
        reduction='none'          # keep per-sample so we can apply focal weight
    )
    # Step 2: p_t = exp(-CE) converts CE loss back to probability of true class
    pt = torch.exp(-ce_per_sample)
    # Step 3: focal weight (1 - p_t)^gamma — shrinks loss for easy examples
    focal_weight = (1.0 - pt) ** gamma
    # Step 4: final focal loss (mean over batch)
    return (focal_weight * ce_per_sample).mean()


# =============================================================================
# JOINT MULTI-TASK LOSS  [updated to use focal loss on multi-class head]
# =============================================================================

def compute_joint_loss(bin_logits, multi_logits, node_logits,
                       y_bin, y_multi, y_node,
                       bin_w, multi_w):

    # Binary head: plain cross-entropy with label smoothing (already balanced)
    bin_loss = F.cross_entropy(
        bin_logits, y_bin, weight=bin_w, label_smoothing=0.05)

    # Multi-class head: focal loss — focuses gradient on hard/rare examples
    # [FIX 4] replaces F.cross_entropy(multi_logits, y_multi, ...)
    multi_loss = focal_loss_with_smoothing(
        multi_logits, y_multi, weight=multi_w, gamma=2.0, label_smoothing=0.05)

    # Per-node auxiliary head: plain cross-entropy (binary, already balanced)
    node_loss = 0
    if y_node is not None:
        node_loss = F.cross_entropy(node_logits, y_node)

    total = bin_loss + 0.7 * multi_loss + 0.2 * node_loss
    return total, bin_loss, multi_loss, node_loss


print('ENS weighting + focal loss + joint loss defined.  [Fix 4 active]')


In [ ]:
# =============================================================================
# CLASS-SPECIFIC REPLAY BUFFER
# Guarantees at least min_per_class samples for EVERY attack type.
# Prevents catastrophic forgetting of rare classes (MITM, Ransomware).
# =============================================================================

class ClassSpecificReplayBuffer:
    """
    Per-class reservoir sampling with minimum-per-class guarantee.
    Each of the NUM_CLASSES classes gets its own reservoir of size min_per_class.
    An additional general reservoir catches overflow.
    """

    def __init__(self, num_classes, min_per_class=50, extra_capacity=1000):
        self.num_classes    = num_classes
        self.min_per_class  = min_per_class
        self.extra_capacity = extra_capacity

        self.class_bufs  = {c: [] for c in range(num_classes)}
        self.class_seen  = {c: 0  for c in range(num_classes)}
        self.extra_buf   = []
        self.extra_seen  = 0

    # ------------------------------------------------------------------
    def add(self, graph):
        c = int(graph.y_multi.item())
        self.class_seen[c] += 1
        n = self.class_seen[c]

        # Per-class reservoir
        if len(self.class_bufs[c]) < self.min_per_class:
            self.class_bufs[c].append(graph)
        else:
            idx = np.random.randint(0, n)
            if idx < self.min_per_class:
                self.class_bufs[c][idx] = graph

        # General overflow reservoir
        self.extra_seen += 1
        if len(self.extra_buf) < self.extra_capacity:
            self.extra_buf.append(graph)
        else:
            idx = np.random.randint(0, self.extra_seen)
            if idx < self.extra_capacity:
                self.extra_buf[idx] = graph

    # ------------------------------------------------------------------
    def sample(self, n):
        pool = self.get_all()
        if not pool:
            return []
        n = min(n, len(pool))
        idxs = np.random.choice(len(pool), n, replace=False)
        return [pool[i] for i in idxs]

    def get_all(self):
        out = []
        for c in range(self.num_classes):
            out.extend(self.class_bufs[c])
        out.extend(self.extra_buf)
        return out

    def per_class_counts(self):
        return {id2attack.get(c, str(c)): len(self.class_bufs[c])
                for c in range(self.num_classes)}

    def __len__(self):
        return (sum(len(b) for b in self.class_bufs.values())
                + len(self.extra_buf))


print('ClassSpecificReplayBuffer defined.')


In [ ]:
# =============================================================================
# ELASTIC WEIGHT CONSOLIDATION (EWC) — lambda reduced to 150
# Lower lambda allows sufficient adaptation to new/evolving attack patterns
# while retaining previously learned knowledge.
# =============================================================================

class EWC:
    def __init__(self, model, dataloader, device, lambda_ewc=150):
        self.model      = model
        self.device     = device
        self.lambda_ewc = lambda_ewc
        self.params     = {}
        self.fisher     = {}
        self._compute_fisher(dataloader)

    def _compute_fisher(self, dataloader):
        self.model.eval()
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.params[name] = param.data.clone()
                self.fisher[name] = torch.zeros_like(param)

        n_samples = 0
        for data in dataloader:
            data = data.to(self.device)
            self.model.zero_grad()
            # Use binary head output for Fisher
            bin_logits, _, _, _, _ = self.model(data)
            log_probs = F.log_softmax(bin_logits, dim=1)
            loss = F.nll_loss(log_probs, data.y_bin.view(-1))
            loss.backward()
            for name, param in self.model.named_parameters():
                if param.requires_grad and param.grad is not None:
                    self.fisher[name] += param.grad.data ** 2
            n_samples += data.y_bin.size(0)

        for name in self.fisher:
            self.fisher[name] /= max(n_samples, 1)

    def penalty(self):
        loss = 0.0
        for name, param in self.model.named_parameters():
            if name in self.fisher:
                loss += (self.fisher[name]
                         * (param - self.params[name]) ** 2).sum()
        return self.lambda_ewc * loss


print('EWC (lambda=150) defined.')


In [ ]:

# =============================================================================
# DRIFT DETECTOR — unchanged logic, used for triggering adaptation
# =============================================================================

class DriftDetector:
    def __init__(self, window_size=100, threshold=0.1,
                 min_samples=50, sensitivity=2.0):
        self.window_size  = window_size
        self.threshold    = threshold
        self.min_samples  = min_samples
        self.sensitivity  = sensitivity
        self.reference    = None
        self.drift_mag    = 0.0

    def update_reference(self, embeddings):
        if torch.is_tensor(embeddings):
            embeddings = embeddings.detach().cpu().numpy()
        self.reference = {
            'mean': np.mean(embeddings, axis=0),
            'std' : np.std(embeddings,  axis=0) + 1e-8
        }

    def check_drift(self, new_emb):
        if self.reference is None:
            return False, 0.0
        if torch.is_tensor(new_emb):
            new_emb = new_emb.detach().cpu().numpy()
        if len(new_emb) < self.min_samples:
            return False, 0.0
        cur_mean = np.mean(new_emb, axis=0)
        cur_std  = np.std(new_emb,  axis=0) + 1e-8
        shift    = np.abs(cur_mean - self.reference['mean']) / self.reference['std']
        self.drift_mag = float(np.mean(shift))
        var_drift = float(np.mean(np.abs(np.log(cur_std / self.reference['std']))))
        total = self.drift_mag + var_drift
        return total > self.threshold * self.sensitivity, total


print('DriftDetector defined.')


In [ ]:
# =============================================================================
# ENHANCED GRAPH BUILDER  (updated for TGN framework)
# Key changes vs original:
#  1. node_ids tensor  : IPv4 hashes for TGN memory lookup
#  2. y_node tensor    : per-node binary labels for auxiliary loss
#  3. multi-label      : majority-attack labelling (not severity-priority)
#                        reduces noisy window labels from mixed traffic
#  4. attack_dist      : stores per-class proportions in the window
# =============================================================================

class EnhancedGraphBuilder:
    """
    Builds PyG Data graphs from temporal flow snapshots.
    Stores all metadata as tensors so they survive DataLoader batching.
    """

    def __init__(self, node_features, edge_features,
                 use_edge_types=True, use_temporal_encoding=True,
                 structural_features=True, temporal_dim=8,
                 edge_decay_factor=0.1, benign_label='benign'):
        self.node_features       = list(node_features)
        self.edge_features       = list(edge_features)
        self.use_edge_types      = use_edge_types
        self.use_temporal_enc    = use_temporal_encoding
        self.use_structural      = structural_features
        self.temporal_dim        = int(temporal_dim)
        self.edge_decay          = float(edge_decay_factor)
        self.benign_label        = benign_label
        self.protocol_to_type    = {6: 0, 17: 1}   # TCP=0, UDP=1, Other=2

    # ------------------------------------------------------------------
    def _edge_type(self, protocol):
        try:
            return self.protocol_to_type.get(int(protocol), 2)
        except Exception:
            return 2

    # ------------------------------------------------------------------
    def _structural_features(self, node_map, df):
        n = len(node_map)
        S = np.zeros((n, 6), dtype=np.float32)
        og = df.groupby('IPV4_SRC_ADDR')
        ig = df.groupby('IPV4_DST_ADDR')
        out_deg  = og.size();              in_deg   = ig.size()
        udst     = og['IPV4_DST_ADDR'].nunique()
        usrc     = ig['IPV4_SRC_ADDR'].nunique()
        udport   = og['L4_DST_PORT'].nunique() if 'L4_DST_PORT' in df.columns else og.size() * 0
        usport   = ig['L4_SRC_PORT'].nunique() if 'L4_SRC_PORT' in df.columns else ig.size() * 0
        for ip, idx in node_map.items():
            S[idx,0] = float(out_deg.get(ip, 0))
            S[idx,1] = float(in_deg.get(ip,  0))
            S[idx,2] = float(udst.get(ip,    0))
            S[idx,3] = float(usrc.get(ip,    0))
            S[idx,4] = float(udport.get(ip,  0))
            S[idx,5] = float(usport.get(ip,  0))
        return S

    # ------------------------------------------------------------------
    def _temporal_encoding(self, n_nodes):
        dim = self.temporal_dim
        if dim <= 0:
            return np.zeros((n_nodes, 0), dtype=np.float32)
        pos = np.arange(n_nodes, dtype=np.float32)[:, None]
        div = np.exp(np.arange(0, dim, 2, dtype=np.float32)
                     * -(np.log(10000.0) / dim))
        pe = np.zeros((n_nodes, dim), dtype=np.float32)
        pe[:, 0::2] = np.sin(pos * div)
        pe[:, 1::2] = np.cos(pos * div)
        return pe

    # ------------------------------------------------------------------
    def _edge_weights(self, df):
        if 'FLOW_START_MILLISECONDS' not in df.columns:
            return np.ones(len(df), dtype=np.float32)
        ts = df['FLOW_START_MILLISECONDS'].values.astype(np.float64)
        if ts.max() == ts.min():
            return np.ones(len(ts), dtype=np.float32)
        norm = (ts - ts.min()) / (ts.max() - ts.min())
        return np.exp(-self.edge_decay * (1.0 - norm)).astype(np.float32)

    # ------------------------------------------------------------------
    def _label_window(self, df):
        """
        Multi-label strategy  [FIX 3b: priority labelling for rare classes]
          y_bin   = 0 if all benign, else 1
          y_multi = majority attack type among ATTACK flows only,
                    BUT scanning / ransomware are promoted to window label
                    whenever they appear at all — even as a minority.

        WHY priority labelling?
          With majority-vote, a window containing 90 benign + 10 scanning
          flows was labelled 'benign', training the model to output 'benign'
          for exactly the mixed-traffic signature that scanning produces.
          Priority labelling ensures the model sees and learns the rare
          attack signal even when it is numerically dominated by benign
          or other attack flows in the same window.
        """
        labels = df['Attack'].astype(str).str.lower()
        attack_mask = labels != self.benign_label
        y_bin = int(attack_mask.any())

        if y_bin == 0:
            y_multi_str = self.benign_label
        else:
            attack_counts = labels[attack_mask].value_counts()

            # FIX 3b: promote these classes to window label whenever present,
            # regardless of their count relative to other attack types.
            # Order matters: ransomware takes priority over scanning if both
            # appear in the same window (extremely rare but handled cleanly).
            PRIORITY_CLASSES = ('ransomware', 'scanning')
            y_multi_str = None
            for cls in PRIORITY_CLASSES:
                if cls in attack_counts.index:
                    y_multi_str = cls
                    break

            # Fall back to majority vote for all other attack types
            if y_multi_str is None:
                y_multi_str = attack_counts.idxmax()

        return y_bin, y_multi_str, attack_mask

    # ------------------------------------------------------------------
    def build_graph(self, df_snap, window_id=0, global_t0=None):
        if len(df_snap) == 0:
            return None

        # ---- node map ----
        ips = pd.concat([
            df_snap['IPV4_SRC_ADDR'],
            df_snap['IPV4_DST_ADDR']
        ]).astype(str).unique()
        node_map = {ip: i for i, ip in enumerate(ips)}
        n_nodes  = len(node_map)

        # ---- node_ids: IPv4 integer hashes (for TGN memory lookup) ----
        node_ids_np = np.array([ip_to_int(ip) for ip in ips], dtype=np.int64)

        # ---- edges ----
        src_idx = df_snap['IPV4_SRC_ADDR'].astype(str).map(node_map).values
        dst_idx = df_snap['IPV4_DST_ADDR'].astype(str).map(node_map).values
        edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)

        if self.use_edge_types and 'PROTOCOL' in df_snap.columns:
            edge_type = torch.tensor(
                df_snap['PROTOCOL'].apply(self._edge_type).values, dtype=torch.long)
        else:
            edge_type = torch.zeros(len(src_idx), dtype=torch.long)

        # ---- node features ----
        tmp = df_snap.copy()
        for c in self.node_features:
            if c not in tmp.columns:
                tmp[c] = 0.0
        tmp[self.node_features] = tmp[self.node_features].fillna(0)

        X_stat = np.zeros((n_nodes, len(self.node_features)), dtype=np.float32)
        for grp_col, mult in [('IPV4_SRC_ADDR', 1), ('IPV4_DST_ADDR', 1)]:
            agg = tmp.groupby(grp_col)[self.node_features].mean()
            for ip in agg.index.astype(str):
                if ip in node_map:
                    X_stat[node_map[ip]] += agg.loc[ip].values.astype(np.float32)
        X_stat /= 2.0

        parts = [X_stat]
        if self.use_structural:
            parts.append(self._structural_features(node_map, df_snap))
        if self.use_temporal_enc and self.temporal_dim > 0:
            parts.append(self._temporal_encoding(n_nodes))
        X = np.concatenate(parts, axis=1) if len(parts) > 1 else parts[0]

        # ---- edge features ----
        for c in self.edge_features:
            if c not in df_snap.columns:
                df_snap = df_snap.copy()
                df_snap[c] = 0.0
        ea_base = df_snap[self.edge_features].fillna(0).values.astype(np.float32)
        ew      = self._edge_weights(df_snap).reshape(-1, 1)
        edge_attr = np.concatenate([ea_base, ew], axis=1)

        # ---- labels ----
        y_bin, y_multi_str, attack_mask = self._label_window(df_snap)

        # Per-node binary labels (1 = node participates in ≥1 attack flow)
        y_node_np = np.zeros(n_nodes, dtype=np.int64)
        if y_bin == 1 and 'Attack' in df_snap.columns:
            atk_rows = df_snap[attack_mask]
            for _, row in atk_rows.iterrows():
                s = str(row['IPV4_SRC_ADDR'])
                d = str(row['IPV4_DST_ADDR'])
                if s in node_map: y_node_np[node_map[s]] = 1
                if d in node_map: y_node_np[node_map[d]] = 1

        # Attack distribution in window (for analysis)
        dist = np.zeros(NUM_CLASSES, dtype=np.float32)
        if 'Attack' in df_snap.columns:
            for a, cnt in df_snap['Attack'].str.lower().value_counts().items():
                if a in attack2id:
                    dist[attack2id[a]] = cnt / len(df_snap)

        attack_ratio = float(attack_mask.mean())

        # ---- build Data object — ALL metadata stored as tensors ----
        data = Data(
            x          = torch.tensor(X,          dtype=torch.float32),
            edge_index = edge_index,
            edge_attr  = torch.tensor(edge_attr,  dtype=torch.float32),
            edge_type  = edge_type,
            y_bin      = torch.tensor([y_bin],    dtype=torch.long),
            y_multi    = torch.tensor([safe_attack_to_id(y_multi_str)], dtype=torch.long),
            y_node     = torch.tensor(y_node_np,  dtype=torch.long),
            node_ids   = torch.tensor(node_ids_np, dtype=torch.long),
            window_id  = torch.tensor([int(window_id)], dtype=torch.long),
            attack_dist= torch.tensor(dist,       dtype=torch.float32),
            attack_ratio=torch.tensor([attack_ratio], dtype=torch.float32),
            n_flows    = torch.tensor([len(df_snap)],  dtype=torch.long),
            num_nodes  = n_nodes,
        )
        return data


# ------------------------------------------------------------------
def clean_graphs(graphs):
    """
    Keep only tensor attributes so DataLoader can batch correctly.
    All key attributes (y_bin, y_multi, y_node, node_ids, window_id, ...)
    are stored as tensors in build_graph, so they are preserved.
    """
    KEEP = {'x','edge_index','edge_attr','edge_type','y_bin','y_multi',
            'y_node','node_ids','window_id','attack_dist','attack_ratio','n_flows'}
    cleaned = []
    for g in graphs:
        new_g = Data()
        for key in KEEP:
            if hasattr(g, key):
                val = getattr(g, key)
                if torch.is_tensor(val):
                    setattr(new_g, key, val)
        new_g.num_nodes = int(g.x.size(0))
        cleaned.append(new_g)
    return cleaned


print('EnhancedGraphBuilder + clean_graphs defined.')
print(f'node_ids tensor   : stores IPv4 hashes -> TGN memory key')
print(f'y_node tensor     : per-node binary labels -> auxiliary loss')
print(f'y_multi labelling : majority-attack-type (multi-label)')


In [ ]:
def iter_time_windows(df, time_col='FLOW_START_MILLISECONDS',
                       window_ms=1000, stride_ms=1000):
    """
    Two-pointer sliding window iterator. Yields (window_id, df_snapshot).
    Assumes df is sorted by time_col (chronological order).
    Default window=1000ms (1s) reduces mixed-traffic label noise vs 5000ms.
    """
    t  = df[time_col].values
    t0, t1 = t.min(), t.max()
    start, wid = t0, 0
    left = right = 0
    n = len(df)
    while start <= t1:
        end = start + window_ms
        while left  < n and t[left]  < start: left  += 1
        while right < n and t[right] < end:   right += 1
        if right > left:
            yield wid, df.iloc[left:right]
        start += stride_ms
        wid   += 1


In [ ]:
# =============================================================================
# WINDOW CONFIGURATION  [FIX 3: wider windows for scanning & ransomware]
#
# WHY 5000ms instead of 1000ms?
#   - Scanning is a *temporal* attack: a host probes many different IPs/ports
#     over time. In a 1-second snapshot, a scanner might hit only 2-3 targets,
#     which is indistinguishable from normal background traffic. The scanning
#     behavioural signature (many unique destinations, low bytes/flow) only
#     accumulates over 5-30 seconds.
#   - Ransomware produces sustained high-throughput encrypted bursts. A 1-second
#     slice may catch only the leading edge, which alone resembles a normal
#     large file transfer. A 5-second window captures enough of the burst to
#     expose the pattern.
#   - The majority-vote window label (used in _label_window) benefits from more
#     flows per window: rare attack flows that were previously a minority in a
#     1-second window are now the clear majority in a 5-second window.
#
# WHY stride = 2500ms (50% overlap)?
#   - 50% overlap ensures that an attack starting in the second half of one
#     window is also fully captured in the next, preventing boundary attacks
#     from being split and mislabelled as benign.
#   - Trade-off: ~2x more graphs than non-overlapping; manageable on GPU.
#     If memory is constrained, set STRIDE_MS = WINDOW_MS for no overlap.
# =============================================================================
WINDOW_MS = 5000   # 5-second windows  (was 1000ms)
STRIDE_MS  = 2500  # 50% overlap stride (was 1000ms)

print(f'Window size : {WINDOW_MS} ms  (FIX 3: increased from 1000ms)')
print(f'Stride      : {STRIDE_MS} ms  (50% overlap to catch boundary attacks)')


In [ ]:
# =============================================================================
# BUILD GRAPHS
# =============================================================================
builder = EnhancedGraphBuilder(
    node_features       = NODE_FEATURES,
    edge_features       = EDGE_FEATURES,
    use_edge_types      = True,
    use_temporal_encoding = True,
    structural_features = True,
    temporal_dim        = 8,
    edge_decay_factor   = 0.1,
)

raw_graphs = []
for wid, snap in tqdm(
        iter_time_windows(df, window_ms=WINDOW_MS, stride_ms=STRIDE_MS),
        desc='Building graphs'):
    g = builder.build_graph(snap, window_id=wid)
    if g is not None:
        raw_graphs.append(g)

graphs = clean_graphs(raw_graphs)
del raw_graphs

print(f'\nTotal graphs built : {len(graphs)}')
g0 = graphs[0]
print(f'Node feature dim   : {g0.x.shape[1]}')
print(f'Edge feature dim   : {g0.edge_attr.shape[1]}')
print(f'node_ids present   : {hasattr(g0, "node_ids")}')
print(f'y_node present     : {hasattr(g0, "y_node")}')
print(f'y_multi label eg   : {id2attack[int(g0.y_multi.item())]}')

IN_DIM   = g0.x.shape[1]
EDGE_DIM = g0.edge_attr.shape[1]
print(f'\nIN_DIM={IN_DIM}  EDGE_DIM={EDGE_DIM}')


In [ ]:
import os
import torch

save_dir = os.path.join("data", "processed", "nf_ton_iot_v3")

os.makedirs(save_dir, exist_ok=True)

print("Saving .pt files to:", save_dir)

In [ ]:
save_path = os.path.join(save_dir, "graphs_dataset.pt")

torch.save(graphs, save_path)

print(f"Saved {len(graphs)} graphs to:", save_path)

In [ ]:
dataset = {
    "graphs": graphs,
    "num_graphs": len(graphs),
    "node_dim": graphs[0].x.shape[1],
    "edge_dim": graphs[0].edge_attr.shape[1]
}

torch.save(dataset, os.path.join(save_dir, "graphs_with_meta.pt"))

In [ ]:
# =============================================================================
# TGN-AUGMENTED JOINT MULTI-TASK IDS MODEL
# Architecture:
#  1. TGN memory lookup    : augment node features with persistent memory
#  2. Memory GRU updater   : GRUCell(hidden_dim -> memory_dim)
#  3. Input projection     : [node_feat + memory] -> hidden_dim
#  4. Multi-layer GAT      : edge-aware message passing with residuals
#  5. SAGPool              : topology-aware graph-level pooling
#  6. Binary head          : benign vs attack (2 classes)
#  7. Attack-type head     : 10-class multiclass
#  8. Node auxiliary head  : per-node binary (participates in attack?)
# All three heads share the same encoder -> joint gradient flow.
# =============================================================================

class TGNJointIDS(nn.Module):

    def __init__(self, in_dim, hidden_dim, num_classes,
                 memory_dim=64, num_layers=3, num_heads=4,
                 dropout=0.3, edge_dim=5, sagpool_ratio=0.5):
        super().__init__()
        self.memory_dim  = memory_dim
        self.hidden_dim  = hidden_dim
        self.num_classes = num_classes

        # --- TGN GRU memory updater ---
        # input  = node embedding (hidden_dim)
        # hidden = current memory  (memory_dim)
        self.memory_updater = nn.GRUCell(
            input_size=hidden_dim, hidden_size=memory_dim)

        # --- Input projection: node_feat + memory -> hidden ---
        self.input_proj = nn.Sequential(
            nn.Linear(in_dim + memory_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )

        # --- GAT message-passing layers ---
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(GATConv(
                hidden_dim,
                hidden_dim // num_heads,
                heads=num_heads,
                dropout=dropout,
                edge_dim=edge_dim,
                add_self_loops=False
            ))
            self.norms.append(nn.LayerNorm(hidden_dim))

        # --- SAGPool: keeps topology-discriminating nodes ---
        self.sagpool = SAGPooling(hidden_dim, ratio=sagpool_ratio)

        # --- Post-pool MLP ---
        self.pool_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # --- Binary detection head ---
        self.binary_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2)
        )

        # --- Attack-type classification head ---
        self.multi_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

        # --- Per-node auxiliary head ---
        self.node_aux_head = nn.Linear(hidden_dim, 2)

        self.dropout = nn.Dropout(dropout)

    # ------------------------------------------------------------------
    def encode(self, data, node_memory=None):
        """
        Compute node embeddings (with TGN memory) and graph embedding (SAGPool).
        Returns: node_emb [N, H], graph_emb [B, H], node_ids list|None
        """
        x         = data.x.float()
        edge_index= data.edge_index
        edge_attr = data.edge_attr.float() if data.edge_attr is not None else None
        batch     = data.batch

        # --- TGN memory augmentation ---
        if node_memory is not None and hasattr(data, 'node_ids'):
            nids = data.node_ids.cpu().tolist()
            mem  = node_memory.get(nids, x.device)   # [N, memory_dim]
        else:
            nids = None
            mem  = torch.zeros(x.size(0), self.memory_dim, device=x.device)

        x = torch.cat([x, mem], dim=-1)              # [N, in_dim + memory_dim]

        # --- Input projection ---
        x = self.input_proj(x)                       # [N, hidden_dim]

        # --- GAT with residual connections ---
        for conv, norm in zip(self.convs, self.norms):
            x_res = x
            x = conv(x, edge_index, edge_attr=edge_attr)
            x = F.elu(x)
            x = norm(x + x_res)
            x = self.dropout(x)

        node_emb = x   # [N, hidden_dim]

        # --- SAGPool: topology-aware graph-level representation ---
        try:
            x_pool, _, _, batch_pool, _, _ = self.sagpool(
                x, edge_index, edge_attr, batch=batch)
            graph_emb = global_mean_pool(x_pool, batch_pool)  # [B, H]
        except Exception:
            # Fallback to global mean pool if SAGPool fails (tiny graphs)
            graph_emb = global_mean_pool(x, batch)

        graph_emb = self.pool_proj(graph_emb)        # [B, hidden_dim]
        return node_emb, graph_emb, nids

    # ------------------------------------------------------------------
    def forward(self, data, node_memory=None):
        """
        Returns:
          bin_logits   [B, 2]
          multi_logits [B, C]
          node_logits  [N, 2]
          node_emb     [N, H]   (for TGN memory update)
          node_ids     list[int] or None
        """
        node_emb, graph_emb, nids = self.encode(data, node_memory)
        bin_logits   = self.binary_head(graph_emb)   # [B, 2]
        multi_logits = self.multi_head(graph_emb)    # [B, C]
        node_logits  = self.node_aux_head(node_emb)  # [N, 2]
        return bin_logits, multi_logits, node_logits, node_emb, nids

    # ------------------------------------------------------------------
    def get_graph_embedding(self, data, node_memory=None):
        """Graph-level embedding for drift detection."""
        _, graph_emb, _ = self.encode(data, node_memory)
        return graph_emb


print('TGNJointIDS defined.')


In [ ]:
# =============================================================================
# FIXED: EARLY STOPPING
# =============================================================================
class EarlyStopping:
    # [EPOCH FIX] patience raised 15→20 to match the increased epoch budget
    # and wider windows that produce noisier per-epoch validation loss.
    def __init__(self, patience=20, min_delta=5e-4, restore_best=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best = restore_best
        self.best_loss = float('inf')
        self.counter = 0
        self.best_state = None
        self.stop = False

    def step(self, val_loss, model):
        # Check if loss improved significantly
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            if self.restore_best:
                # FIX: Ensure state is cloned and stored
                self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore(self, model, device):
        if self.restore_best and self.best_state is not None:
            # FIX: Load back to the active device
            model.load_state_dict(self.best_state)
            model.to(device)
            print("Restored best model state.")

# =============================================================================
# FIXED: JOINT TRAINER
# =============================================================================
class JointTrainer:
    def __init__(self, model, tgn_memory, device,
                 bin_weights, multi_weights,
                 lr=1e-3, ewc_lambda=150,
                 replay_buffer=None, drift_detector=None,
                 min_lr=1e-5, update_interval=20, replay_batch=64):

        self.model          = model.to(device)
        self.tgn_memory     = tgn_memory
        self.device         = device
        self.bin_weights    = bin_weights.to(device)
        self.multi_weights  = multi_weights.to(device)
        self.lr             = lr
        self.min_lr         = min_lr
        self.update_interval= update_interval
        self.replay_batch   = replay_batch

        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        # [EPOCH FIX] patience raised 5→10: wider windows (Fix 3) produce fewer
        # graphs/epoch so loss oscillations are larger; LR must not collapse early.
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=10, min_lr=min_lr)

        self.replay_buffer  = replay_buffer
        self.drift_detector = drift_detector
        self.ewc            = None

        self.train_losses, self.val_losses = [], []
        self.train_bin_accs, self.val_bin_accs = [], []
        self.train_multi_accs, self.val_multi_accs = [], []

    def _step_batch(self, data):
        data = data.to(self.device)
        self.model.train()
        self.optimizer.zero_grad()

        # Forward pass
        bin_l, multi_l, node_l, node_emb, nids = self.model(data, self.tgn_memory)

        y_bin   = data.y_bin.view(-1)
        y_multi = data.y_multi.view(-1)
        y_node  = data.y_node.view(-1) if hasattr(data, 'y_node') else None

        loss, _, _, _ = compute_joint_loss(
            bin_l, multi_l, node_l, y_bin, y_multi, y_node,
            self.bin_weights, self.multi_weights)

        if self.ewc is not None:
            loss = loss + self.ewc.penalty()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()

        # FIX: DETACH node embeddings before memory update to prevent gradient leaks
        if nids is not None:
            self.tgn_memory.update_with_gru(
                nids, node_emb.detach(), # CRITICAL: .detach() added
                self.model.memory_updater, self.device)

        bin_acc   = (bin_l.argmax(1) == y_bin).float().mean().item()
        multi_acc = (multi_l.argmax(1) == y_multi).float().mean().item()
        return loss.item(), bin_acc, multi_acc

    def initial_training(self, train_graphs, val_graphs, epochs=50, batch_size=16):

        train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=False)
        val_loader   = DataLoader(val_graphs, batch_size=batch_size, shuffle=False)
    
        best = 0
    
        for epoch in range(epochs):
            self.tgn_memory.reset()
    
            total_loss = 0
            correct = 0
            total = 0
    
            for data in train_loader:
                loss, preds = self._step(data)
    
                total_loss += loss * data.y_multi.size(0)
                correct += (preds == data.y_multi.to(self.device)).sum().item()
                total += data.y_multi.size(0)
    
            train_loss = total_loss / max(total, 1)
            train_acc = correct / max(total, 1)
    
            val_loss, val_acc = self.evaluate(val_loader)
    
            print(f"Epoch {epoch:02d} | ValLoss {val_loss:.4f} | ValAcc {val_acc:.4f}")
    
            if val_acc > best:
                best = val_acc
                best_state = self.model.state_dict()
    
        self.model.load_state_dict(best_state)

In [ ]:
# =============================================================================
# FIXED: STRATIFIED TEMPORAL SPLIT
# =============================================================================
from sklearn.model_selection import train_test_split

def stratified_chronological_split(graphs, train_ratio=0.60, val_ratio=0.15):
    """
    Ensures every attack class is represented in Train, Val, and Stream
    while maintaining as much temporal order as possible.
    """
    labels = [int(g.y_multi.item()) for g in graphs]
    
    # 1. First, split off the Training set (Stratified)
    # This ensures 'xss', 'mitm', etc., are present in Train
    train_idx, temp_idx = train_test_split(
        np.arange(len(graphs)), 
        train_size=train_ratio, 
        stratify=labels, 
        random_state=42
    )
    
    # 2. Split the remainder into Val and Stream
    temp_labels = [labels[i] for i in temp_idx]
    val_rel_ratio = val_ratio / (1 - train_ratio)
    
    val_idx, stream_idx = train_test_split(
        temp_idx, 
        train_size=val_rel_ratio, 
        stratify=temp_labels, 
        random_state=42
    )
    
    # 3. CRITICAL: Re-sort each split CHRONOLOGICALLY
    # TGN memory MUST process graphs in time order within each set
    train_graphs  = sorted([graphs[i] for i in train_idx], key=lambda g: g.t_start if hasattr(g, 't_start') else 0)
    val_graphs    = sorted([graphs[i] for i in val_idx], key=lambda g: g.t_start if hasattr(g, 't_start') else 0)
    stream_graphs = sorted([graphs[i] for i in stream_idx], key=lambda g: g.t_start if hasattr(g, 't_start') else 0)
    
    return train_graphs, val_graphs, stream_graphs

# Apply the new split
train_graphs, val_graphs, stream_graphs = stratified_chronological_split(graphs)

# --- Verify the new counts ---
print(f"New Train Classes: {Counter([id2attack[int(g.y_multi.item())] for g in train_graphs])}")
print(f"New Stream Classes: {Counter([id2attack[int(g.y_multi.item())] for g in stream_graphs])}")

In [ ]:
# =============================================================================
# COMPUTE ENS CLASS WEIGHTS  [FIX 1: use original dataset frequencies]
#
# THE BUG THIS FIXES:
#   After balancing all classes to 120,000 graphs each, the training-set
#   counts fed to compute_ens_weights were approximately equal for every
#   class. ENS produces near-identical weights (~1.0) for all classes,
#   so the model received no extra gradient signal for hard/rare classes.
#   Ransomware (originally 3,971 flows) was penalised identically to DDoS
#   (originally 828,252 flows) — completely defeating class weighting.
#
# THE FIX:
#   Supply the *original* pre-balancing flow counts to compute_ens_weights.
#   These reflect how naturally rare each class truly is in real traffic,
#   which is exactly the signal ENS weighting is designed to amplify.
#   Ransomware gets a weight ~200x higher than DDoS before capping.
# =============================================================================

# --- Binary weights (stable; computed from train-set as before) ---
bin_counts = np.array(
    [Counter([int(g.y_bin.item()) for g in train_graphs])[i] for i in range(2)],
    dtype=np.float64
)
bin_weights = compute_ens_weights(bin_counts + 1, beta=0.999, device=device)
print(f'Binary counts : benign={bin_counts[0]:.0f}  attack={bin_counts[1]:.0f}')
print(f'Binary ENS weights: {bin_weights.cpu().numpy().round(4)}')

# --- Multiclass weights: FIX 1 — original dataset frequencies ---
# These are the raw flow counts from the NF-ToN-IoT-v3 dataset BEFORE
# any balancing. Update these numbers if you switch datasets.
ORIGINAL_CLASS_COUNTS = {
    'benign':     3358444,
    'ddos':        828252,
    'xss':         566887,
    'password':    318956,
    'scanning':    271795,
    'backdoor':    203384,
    'injection':    76356,
    'dos':          40692,
    'mitm':          6013,
    'ransomware':    3971,
}

original_class_counts = np.array(
    [ORIGINAL_CLASS_COUNTS[a] for a in ATTACK_CLASSES], dtype=np.float64
)

print(f'\nOriginal flow counts used for ENS weighting:')
for i, a in enumerate(ATTACK_CLASSES):
    print(f'  {a:<15} {original_class_counts[i]:>10,.0f}')

# Compute ENS weights from original counts
multi_weights = compute_ens_weights(original_class_counts, beta=0.9999, device=device)

# Cap the weight ratio at 30x to keep training numerically stable.
# Without a cap the raw ransomware/ddos ratio would be ~200x, which
# causes gradient explosions when the model makes an early ransomware error.
max_ratio = 30.0
min_w = multi_weights.min()
multi_weights = torch.clamp(multi_weights, max=min_w * max_ratio)

print(f'\nMulticlass ENS weights (original-freq, capped at {max_ratio:.0f}x):')
for i, a in enumerate(ATTACK_CLASSES):
    print(f'  {a:<15} weight={multi_weights[i].item():.4f}')
print(f'\nWeight spread: {multi_weights.max().item()/multi_weights.min().item():.2f}x  '
      f'(was ~1.0x before Fix 1)')


In [ ]:
# =============================================================================
# FIXED: INITIALIZE TGN + REPLAY BUFFER + TRAINER
# =============================================================================

HIDDEN_DIM  = 192   # increased capacity
MEMORY_DIM  = 96    # stronger temporal memory
NUM_LAYERS  = 3
NUM_HEADS   = 4
DROPOUT     = 0.25  # slightly reduced to improve recall
SAGPOOL_R   = 0.6   # retain more nodes (better recall)

# --- Model ---
model = TGNJointIDS(
    in_dim        = IN_DIM,
    hidden_dim    = HIDDEN_DIM,
    num_classes   = NUM_CLASSES,
    memory_dim    = MEMORY_DIM,
    num_layers    = NUM_LAYERS,
    num_heads     = NUM_HEADS,
    dropout       = DROPOUT,
    edge_dim      = EDGE_DIM,
    sagpool_ratio = SAGPOOL_R
).to(device)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

# --- TGN memory ---
tgn_memory = TGNNodeMemory(memory_dim=MEMORY_DIM)

# --- Class-specific replay buffer ---
# FIX: Change min_per_class to 10. 
# Your 'ransomware' count is 79; if you require 50 to start, 
# the buffer might not activate properly during the early streaming phase.
replay_buffer = ClassSpecificReplayBuffer(
    num_classes    = NUM_CLASSES,
    min_per_class  = 10, 
    extra_capacity = 2000 # Increased to hold more diverse attack patterns
)

# --- Drift detector ---
# FIX: Sensitivity 5.0 is more stable for network traffic.
drift_detector = DriftDetector(window_size=100, threshold=0.15, sensitivity=5.0)

# --- Trainer ---
trainer = JointTrainer(
    model          = model,
    tgn_memory     = tgn_memory,
    device         = device,
    bin_weights    = bin_weights,   # Uses the [1.0, 1.0] weights
    multi_weights  = multi_weights, # Uses the [0.04 to 2.35] stabilized weights
    lr             = 1e-3,
    ewc_lambda     = 150,
    replay_buffer  = replay_buffer,
    drift_detector = drift_detector,
    min_lr         = 1e-5,
    update_interval= 50, # Increased from 20 for better TGN memory stability
    replay_batch   = 64,
)

print('All components initialized with stratified weights and stability fixes.')

In [ ]:
import time
import numpy as np
import torch.nn.functional as F
from torch_geometric.data import Batch
from collections import deque
import random
# =============================================================================
# FIXED: JOINT TRAINER (Consolidated to include all features)
# =============================================================================
class JointTrainer:
    def __init__(self, model, tgn_memory, device,
                 bin_weights, multi_weights,
                 lr=1e-3, ewc_lambda=150,
                 replay_buffer=None, drift_detector=None,
                 min_lr=1e-5, update_interval=20, replay_batch=64,
                 early_stopping_patience=20):

        self.model          = model.to(device)
        self.tgn_memory     = tgn_memory
        self.device         = device
        self.bin_weights    = bin_weights.to(device)
        self.multi_weights  = multi_weights.to(device)
        self.lr             = lr
        self.min_lr         = min_lr
        self.update_interval= update_interval
        self.replay_batch   = replay_batch

        # Early stopping
        self.early_stopping_patience = early_stopping_patience
        self.best_val_loss = float('inf')
        self.early_stop_counter = 0

        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=1e-4
        )

        # LR scheduler patience
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode='min',
            factor=0.5,
            patience=10,
            min_lr=min_lr
        )

        self.replay_buffer = replay_buffer or deque(maxlen=5000)
        self.drift_detector = drift_detector
        self.ewc = None

        self.train_losses, self.val_losses = [], []
        self.train_bin_accs, self.val_bin_accs = [], []
        self.train_multi_accs, self.val_multi_accs = [], []

        self.best_state = None
        self.best_acc = 0.0
    
    
    # ============================================================
    # TRAIN STEP (for a single batch)
    # ============================================================
    def _step(self, data):
        data = data.to(self.device)
        self.model.train()
        self.optimizer.zero_grad()

        # Forward pass
        bin_l, multi_l, node_l, node_emb, nids = self.model(data, self.tgn_memory)

        y_bin   = data.y_bin.view(-1)
        y_multi = data.y_multi.view(-1)
        y_node  = data.y_node.view(-1) if hasattr(data, 'y_node') else None

        loss, _, _, _ = compute_joint_loss(
            bin_l, multi_l, node_l, y_bin, y_multi, y_node,
            self.bin_weights, self.multi_weights)

        if self.ewc is not None:
            loss = loss + self.ewc.penalty()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()

        # FIX: DETACH node embeddings before memory update to prevent gradient leaks
        if nids is not None:
            self.tgn_memory.update_with_gru(
                nids, node_emb.detach(), # CRITICAL: .detach() added
                self.model.memory_updater, self.device)

        bin_acc   = (bin_l.argmax(1) == y_bin).float().mean().item()
        multi_acc = (multi_l.argmax(1) == y_multi).float().mean().item()
        return loss.item(), bin_acc, multi_acc, multi_l.argmax(1) # Return multi_preds for consistency

    # ============================================================
    # EVALUATION (FULL LOSS + ACCURACY)
    # ============================================================
    def evaluate(self, loader):
        self.model.eval()

        total_loss = 0
        correct_multi = 0
        total_samples = 0

        with torch.no_grad():
            for data in loader:
                data = data.to(self.device)

                bin_l, multi_l, node_l, _, _ = self.model(data, self.tgn_memory)

                loss, _, _, _ = compute_joint_loss(
                    bin_l,
                    multi_l,
                    node_l,
                    data.y_bin.view(-1),
                    data.y_multi.view(-1),
                    data.y_node.view(-1) if hasattr(data, 'y_node') else None,
                    self.bin_weights,
                    self.multi_weights
                )

                total_loss += loss.item() * data.num_graphs # Use num_graphs for batch size
                preds_multi = multi_l.argmax(1)
                correct_multi += (preds_multi == data.y_multi.to(self.device)).sum().item()
                total_samples += data.num_graphs # Accumulate total graphs

        avg_loss = total_loss / max(total_samples, 1)
        acc_multi = correct_multi / max(total_samples, 1)

        return avg_loss, acc_multi

    # ============================================================
    # INITIAL TRAINING LOOP
    # This method is for the initial, offline training phase.
    # ============================================================
    def train(self, train_loader, val_loader, epochs=150):
        early_stopping = EarlyStopping(patience=20, min_delta=5e-4) # Use the class defined earlier

        # reset self.tgn_memory for initial training
        self.tgn_memory.reset()

        for epoch in range(epochs):
            self.model.train()
            total_train_loss = 0
            correct_train_multi = 0
            total_train_samples = 0

            for data in train_loader:
                loss, bin_acc, multi_acc, multi_preds = self._step(data) # _step returns multi_preds now
                total_train_loss += loss * data.num_graphs
                correct_train_multi += (multi_preds == data.y_multi.to(self.device)).sum().item()
                total_train_samples += data.num_graphs

            train_loss = total_train_loss / max(total_train_samples, 1)
            train_acc_multi = correct_train_multi / max(total_train_samples, 1)

            val_loss, val_acc_multi = self.evaluate(val_loader)

            self.scheduler.step(val_loss) # Scheduler steps based on validation loss

            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.train_multi_accs.append(train_acc_multi)
            self.val_multi_accs.append(val_acc_multi)

            print(
                f"Epoch {epoch+1:03d} | "
                f"TrainLoss: {train_loss:.4f} | ValLoss: {val_loss:.4f} | "
                f"TrainAcc: {train_acc_multi:.4f} | ValAcc: {val_acc_multi:.4f} | "
                f"LR: {self.optimizer.param_groups[0]['lr']:.6f}"
            )

            early_stopping.step(val_loss, self.model)
            if val_acc_multi > self.best_acc: # Track best model based on multi_acc
                self.best_acc = val_acc_multi
                self.best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
            if early_stopping.stop:
                print(f"Early stopping at epoch {epoch+1}")
                break

        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)
            print(f"\nBest model restored with ValAcc: {self.best_acc:.4f}")

        return {
            "train_loss": self.train_losses,
            "val_loss": self.val_losses,
            "train_acc": self.train_multi_accs,
            "val_acc": self.val_multi_accs,
            "lr": [pg['lr'] for pg in self.optimizer.param_groups] # Store all LRs
        }


    # ============================================================
    # INCREMENTAL UPDATE (for streaming phase)
    # ============================================================
    def incremental_update(self, graphs_batch, n_steps=1):
        """
        Performs incremental training steps on a batch of new graphs.
        Integrates replay buffer and drift detection.
        """
        self.model.train()
        n_graphs_processed = 0

        # Add new graphs to replay buffer
        for g in graphs_batch:
            self.replay_buffer.append(g)

        # Main adaptation loop
        for _ in range(n_steps):
            # Sample from replay buffer
            replayed_graphs = random.sample(
                self.replay_buffer,
                k=min(self.replay_batch, len(self.replay_buffer))
                )
            if not replayed_graphs:
                break
            replayed_batch = Batch.from_data_list(replayed_graphs).to(self.device)

            # Perform one training step
            loss, bin_acc, multi_acc, _ = self._step(replayed_batch)
            n_graphs_processed += replayed_batch.num_graphs

        self.model.eval() # Switch back to eval mode after update
        # print(f"Incremental update: {n_graphs_processed} graphs, Loss: {loss:.4f}")


In [ ]:
import torch

# --- Device ---
device = torch.device('cuda')
print(f"System switched to: {device}")

# Move class weights to the active device
final_multi_weights = multi_weights.to(device)
final_bin_weights   = bin_weights.to(device)

# Re-initialize model with the CORRECTED, higher capacity parameters
model = TGNJointIDS(
    in_dim=IN_DIM,
    hidden_dim=HIDDEN_DIM,    # Use the intended HIDDEN_DIM (192)
    num_classes=NUM_CLASSES,
    memory_dim=MEMORY_DIM,    # Use the intended MEMORY_DIM (96)
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,          # Use the intended DROPOUT (0.25)
    edge_dim=EDGE_DIM,
    sagpool_ratio=SAGPOOL_R   # Use the intended SAGPOOL_R (0.6)
).to(device)

tgn_memory = TGNNodeMemory(memory_dim=MEMORY_DIM) # Use the intended MEMORY_DIM (96)

# Re-initialize trainer
trainer = JointTrainer(
    model=model, tgn_memory=tgn_memory, device=device,
    bin_weights=final_bin_weights, multi_weights=final_multi_weights,
    lr=1e-3
)

In [ ]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=False)
val_loader   = DataLoader(val_graphs, batch_size=32, shuffle=False)

In [ ]:
# [EPOCH FIX] epochs raised 100→150  (see JointTrainer.train docstring)
history = trainer.train(train_loader, val_loader, epochs=150)

## Five-seed robustness and operating-point analysis

In [ ]:
import random
import numpy as np
import torch
import pandas as pd
from torch_geometric.loader import DataLoader

# ============================================================
# 5-SEED REPETITION
# ============================================================

SEEDS = [13, 42, 123, 777, 2026]

all_histories = {}

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ============================================================
# SAME TRAIN / VALIDATION SPLIT FOR ALL SEEDS
# ============================================================

train_loader = DataLoader(
    train_graphs,
    batch_size=32,
    shuffle=False
)

val_loader = DataLoader(
    val_graphs,
    batch_size=32,
    shuffle=False
)


# ============================================================
# CLASS WEIGHTS
# Move once to the selected device
# ============================================================

final_bin_weights = bin_weights.to(device)
final_multi_weights = multi_weights.to(device)


# ============================================================
# RUN FIVE INDEPENDENT TRAININGS
# ============================================================

for seed_idx, seed in enumerate(SEEDS, 1):

    print("\n" + "=" * 80)
    print(
        f"SEED {seed} "
        f"({seed_idx}/{len(SEEDS)})"
    )
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Set random seed
    # --------------------------------------------------------
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Improve reproducibility on GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


    # --------------------------------------------------------
    # 2. NEW MODEL FOR THIS SEED
    # --------------------------------------------------------
    model = TGNJointIDS(
        in_dim=IN_DIM,
        hidden_dim=HIDDEN_DIM,
        num_classes=NUM_CLASSES,
        memory_dim=MEMORY_DIM,
        num_layers=NUM_LAYERS,
        num_heads=NUM_HEADS,
        dropout=DROPOUT,
        edge_dim=EDGE_DIM,
        sagpool_ratio=SAGPOOL_R
    ).to(device)


    # --------------------------------------------------------
    # 3. NEW TGN MEMORY FOR THIS SEED
    # --------------------------------------------------------
    tgn_memory = TGNNodeMemory(
        memory_dim=MEMORY_DIM
    )


    # --------------------------------------------------------
    # 4. NEW TRAINER FOR THIS SEED
    # --------------------------------------------------------
    trainer = JointTrainer(
        model=model,
        tgn_memory=tgn_memory,
        device=device,

        bin_weights=final_bin_weights,
        multi_weights=final_multi_weights,

        lr=1e-3,
        ewc_lambda=150,

        replay_buffer=replay_buffer,
        drift_detector=drift_detector,

        min_lr=1e-5,

        # Original trainer setting
        update_interval=50,
        replay_batch=64,

        # NEW: early stopping
        early_stopping_patience=20
    )


    # --------------------------------------------------------
    # 5. TRAIN FROM SCRATCH
    # Maximum = 150 epochs
    # Early stopping = patience 20
    # --------------------------------------------------------
    history = trainer.train(
        train_loader,
        val_loader,
        epochs=150
    )


    # --------------------------------------------------------
    # 6. SAVE TRAINING HISTORY
    # --------------------------------------------------------
    all_histories[seed] = history


    # --------------------------------------------------------
    # 7. PRINT SUMMARY
    # --------------------------------------------------------
    print("\n" + "-" * 60)
    print(f"Seed {seed} training completed.")

    if hasattr(trainer, 'best_val_loss'):
        print(
            f"Best validation loss: "
            f"{trainer.best_val_loss:.6f}"
        )

    if hasattr(trainer, 'train_losses'):
        print(
            f"Epochs executed: "
            f"{len(trainer.train_losses)} / 150"
        )

    print("-" * 60)


    # --------------------------------------------------------
    # Optional GPU memory cleanup
    # --------------------------------------------------------
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\n" + "=" * 80)
print("ALL 5 SEED TRAININGS COMPLETED")
print("=" * 80)

print(f"Seeds evaluated: {SEEDS}")

In [ ]:
torch.save(
    {
        "seed": 2026,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": trainer.optimizer.state_dict(),
        "history": history
    },
    "TGN_JointIDS_seed_2026.pt"
)

print("Seed 2026 saved.")

In [ ]:
# ==============================================================
# SEED 2026 — POST-TRAINING STREAMING EVALUATION
# NO RETRAINING
# ==============================================================

import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

SEED = 2026

print("=" * 90)
print(f"STREAMING EVALUATION — SEED {SEED}")
print("=" * 90)
print("Device:", device)


# ==============================================================
# 1. LOAD SAVED SEED-2026 MODEL
# ==============================================================

CHECKPOINT = "TGN_JointIDS_seed_2026.pt"

if os.path.exists(CHECKPOINT):

    ckpt = torch.load(
        CHECKPOINT,
        map_location=device
    )

    model.load_state_dict(
        ckpt["model_state_dict"]
    )

    print(f"✓ Loaded saved model: {CHECKPOINT}")

else:

    print(
        "Checkpoint not found, using the model "
        "currently present in memory."
    )


model = model.to(device)
model.eval()


# ==============================================================
# 2. RESET TEMPORAL MEMORY
# ==============================================================

print("\n" + "=" * 90)
print("PHASE 2 — TGN MEMORY WARM-UP")
print("=" * 90)

tgn_memory.reset()

print(
    "Memory immediately after reset:",
    len(tgn_memory.memory)
)


# ==============================================================
# 3. CHRONOLOGICAL MEMORY WARM-UP
# ==============================================================

warmup_loader = DataLoader(
    train_graphs,
    batch_size=16,
    shuffle=False
)

model.eval()

with torch.no_grad():

    for batch_idx, data in enumerate(warmup_loader):

        data = data.to(device)

        bin_logits, \
        multi_logits, \
        node_logits, \
        node_emb, \
        nids = model(
            data,
            tgn_memory
        )

        if nids is not None:

            tgn_memory.update_with_gru(
                nids,
                node_emb.detach(),
                model.memory_updater,
                device
            )

        if (batch_idx + 1) % 500 == 0:

            print(
                f"Warm-up batch {batch_idx + 1:,} | "
                f"memory keys = {len(tgn_memory.memory):,}"
            )


print(
    f"\n✓ Memory warm-up complete."
    f"\n  Stored node memories: {len(tgn_memory.memory):,}"
)


# ==============================================================
# 4. EWC REFERENCE
# ==============================================================

print("\n" + "=" * 90)
print("COMPUTING EWC REFERENCE")
print("=" * 90)

ewc_graphs = train_graphs[:2000]

ewc_loader = DataLoader(
    ewc_graphs,
    batch_size=16,
    shuffle=False
)

trainer.ewc = EWC(
    model,
    ewc_loader,
    device,
    lambda_ewc=150
)

print("✓ EWC reference created.")


# ==============================================================
# 5. STREAMING EVALUATION
# ==============================================================

print("\n" + "=" * 90)
print("PHASE 3 — STREAMING")
print("=" * 90)

UPDATE_INTERVAL = 32

stream_results_2026 = []
latencies_2026 = []
accumulated = []

model.eval()

n_stream = len(stream_graphs)

for i, graph in enumerate(stream_graphs):

    # ----------------------------------------------------------
    # Inference timer
    # ----------------------------------------------------------

    t0 = time.perf_counter()

    with torch.no_grad():

        g_batch = Batch.from_data_list(
            [graph]
        ).to(device)

        bin_logits, \
        multi_logits, \
        _, \
        node_emb, \
        nids = model(
            g_batch,
            tgn_memory
        )

        bin_probs = F.softmax(
            bin_logits,
            dim=1
        )

        multi_probs = F.softmax(
            multi_logits,
            dim=1
        )

        bin_pred = int(
            bin_logits.argmax(
                dim=1
            ).item()
        )

        multi_pred = int(
            multi_logits.argmax(
                dim=1
            ).item()
        )

        bin_conf = float(
            bin_probs.max().item()
        )

        attack_prob = float(
            bin_probs[0, 1].item()
        )

        multi_conf = float(
            multi_probs.max().item()
        )


    # ----------------------------------------------------------
    # Update TGN memory after inference
    # ----------------------------------------------------------

    if nids is not None:

        tgn_memory.update_with_gru(
            nids,
            node_emb.detach(),
            model.memory_updater,
            device
        )


    latency_ms = (
        time.perf_counter() - t0
    ) * 1000.0

    latencies_2026.append(
        latency_ms
    )


    # ----------------------------------------------------------
    # Ground truth
    # ----------------------------------------------------------

    true_bin = int(
        graph.y_bin.item()
    )

    true_multi = int(
        graph.y_multi.item()
    )

    if hasattr(graph, "window_id"):

        wid = int(
            graph.window_id.item()
        )

    else:

        wid = i


    # ----------------------------------------------------------
    # Store output
    # ----------------------------------------------------------

    stream_results_2026.append({

        "window_id":
            wid,

        "bin_true":
            true_bin,

        "bin_pred":
            bin_pred,

        "bin_conf":
            bin_conf,

        "attack_prob":
            attack_prob,

        "multi_true":
            true_multi,

        "multi_pred":
            multi_pred,

        "multi_conf":
            multi_conf,

        "latency_ms":
            latency_ms
    })


    # ----------------------------------------------------------
    # Online adaptation
    # ----------------------------------------------------------

    accumulated.append(
        graph
    )

    if len(accumulated) >= UPDATE_INTERVAL:

        trainer.incremental_update(
            accumulated,
            n_steps=1
        )

        model.eval()

        accumulated = []


    # ----------------------------------------------------------
    # Progress
    # ----------------------------------------------------------

    if (i + 1) % 1000 == 0:

        print(
            f"{i + 1:,}/{n_stream:,} windows | "
            f"memory={len(tgn_memory.memory):,} | "
            f"mean latency="
            f"{np.mean(latencies_2026):.3f} ms"
        )


# Remaining graphs
if len(accumulated) > 0:

    trainer.incremental_update(
        accumulated,
        n_steps=1
    )

    model.eval()


print(
    f"\n✓ Streaming complete: "
    f"{len(stream_results_2026):,} windows"
)


# ==============================================================
# 6. BINARY METRICS
# ==============================================================

y_true_bin = np.array(
    [
        x["bin_true"]
        for x in stream_results_2026
    ]
)

y_pred_bin = np.array(
    [
        x["bin_pred"]
        for x in stream_results_2026
    ]
)


streaming_accuracy = accuracy_score(
    y_true_bin,
    y_pred_bin
)


# Attack-class F1
binary_f1 = f1_score(
    y_true_bin,
    y_pred_bin,
    pos_label=1,
    zero_division=0
)


# Weighted binary F1
binary_weighted_f1 = f1_score(
    y_true_bin,
    y_pred_bin,
    average="weighted",
    zero_division=0
)


cm_bin = confusion_matrix(
    y_true_bin,
    y_pred_bin,
    labels=[0, 1]
)

tn, fp, fn, tp = cm_bin.ravel()


fnr = (
    fn / (fn + tp)
    if (fn + tp) > 0
    else 0.0
)


benign_fpr = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)


# ==============================================================
# 7. MULTICLASS METRICS
# ==============================================================

y_true_multi = np.array(
    [
        x["multi_true"]
        for x in stream_results_2026
    ]
)

y_pred_multi = np.array(
    [
        x["multi_pred"]
        for x in stream_results_2026
    ]
)


# Ground-truth ATTACK windows only
attack_mask = (
    y_true_bin == 1
)

y_true_attack = (
    y_true_multi[
        attack_mask
    ]
)

y_pred_attack = (
    y_pred_multi[
        attack_mask
    ]
)


multiclass_weighted_f1 = f1_score(
    y_true_attack,
    y_pred_attack,
    average="weighted",
    zero_division=0
)


multiclass_macro_f1 = f1_score(
    y_true_attack,
    y_pred_attack,
    average="macro",
    zero_division=0
)


# ==============================================================
# 8. PER-CLASS RECALL
# ==============================================================

def class_recall(
    class_name,
    y_true,
    y_pred
):

    cid = attack2id[
        class_name
    ]

    mask = (
        y_true == cid
    )

    count = int(
        mask.sum()
    )

    if count == 0:

        return np.nan, 0

    recall = float(
        np.mean(
            y_pred[mask] == cid
        )
    )

    return recall, count


mitm_recall, mitm_n = \
    class_recall(
        "mitm",
        y_true_attack,
        y_pred_attack
    )


ransomware_recall, ransomware_n = \
    class_recall(
        "ransomware",
        y_true_attack,
        y_pred_attack
    )


scanning_recall, scanning_n = \
    class_recall(
        "scanning",
        y_true_attack,
        y_pred_attack
    )


# ==============================================================
# 9. LATENCY
# ==============================================================

mean_latency_ms = float(
    np.mean(
        latencies_2026
    )
)


p95_latency_ms = float(
    np.percentile(
        latencies_2026,
        95
    )
)


# ==============================================================
# 10. FINAL RESULT DICTIONARY
# ==============================================================

seed_2026_result = {

    "seed":
        2026,

    "stream_windows":
        len(
            stream_results_2026
        ),

    "streaming_accuracy":
        streaming_accuracy,

    "binary_f1":
        binary_f1,

    "binary_weighted_f1":
        binary_weighted_f1,

    "fnr":
        fnr,

    "benign_fpr":
        benign_fpr,

    "multiclass_weighted_f1":
        multiclass_weighted_f1,

    "multiclass_macro_f1":
        multiclass_macro_f1,

    "mitm_recall":
        mitm_recall,

    "ransomware_recall":
        ransomware_recall,

    "scanning_recall":
        scanning_recall,

    "mean_latency_ms":
        mean_latency_ms,

    "p95_latency_ms":
        p95_latency_ms,

    "memory_keys":
        len(
            tgn_memory.memory
        )
}


# ==============================================================
# 11. PRINT RESULTS
# ==============================================================

print("\n")
print("=" * 90)
print("SEED 2026 — FINAL STREAMING RESULTS")
print("=" * 90)

for key, value in seed_2026_result.items():

    if isinstance(
        value,
        (float, np.floating)
    ):

        print(
            f"{key:<32}: "
            f"{value:.4f}"
        )

    else:

        print(
            f"{key:<32}: "
            f"{value}"
        )


print("\nBinary confusion matrix:")
print(cm_bin)


print(
    "\nAttack-window counts:"
)

print(
    f"MITM       : {mitm_n}"
)

print(
    f"Ransomware : {ransomware_n}"
)

print(
    f"Scanning   : {scanning_n}"
)


# ==============================================================
# 12. SAVE RESULTS
# ==============================================================

pd.DataFrame(
    [seed_2026_result]
).to_csv(
    "seed_2026_streaming_results.csv",
    index=False
)


pd.DataFrame(
    stream_results_2026
).to_csv(
    "seed_2026_streaming_predictions.csv",
    index=False
)


print(
    "\n✓ Saved:"
    "\n  seed_2026_streaming_results.csv"
    "\n  seed_2026_streaming_predictions.csv"
)

In [ ]:
 # =====================================================================
# OPERATING-POINT ANALYSIS (Section 5.6)
# Benign representation + threshold sensitivity + Precision-Recall curve
#
# NO RETRAINING REQUIRED
# =====================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score,
    accuracy_score
)

# =====================================================================
# 1. LOAD SEED-2026 STREAMING PREDICTIONS
# =====================================================================

PRED_FILE = "seed_2026_streaming_predictions.csv"

df_pred = pd.read_csv(PRED_FILE)

print("=" * 100)
print("OPERATING-POINT ANALYSIS (Section 5.6) — ADDITIONAL EXPERIMENT")
print("=" * 100)

print("\nPrediction file:")
print(PRED_FILE)

print("\nColumns:")
print(df_pred.columns.tolist())

print("\nNumber of streaming windows:")
print(len(df_pred))


# =====================================================================
# 2. VERIFY REQUIRED COLUMNS
# =====================================================================

required_cols = [
    "bin_true",
    "bin_pred",
    "attack_prob",
    "multi_true",
    "multi_pred"
]

missing = [
    c for c in required_cols
    if c not in df_pred.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("\n✓ Required columns found.")


# =====================================================================
# 3. WINDOW-LEVEL CLASS COMPOSITION
# =====================================================================

print("\n" + "=" * 100)
print("A. WINDOW-LEVEL CLASS COMPOSITION")
print("=" * 100)


def binary_composition(graph_list, split_name):

    y = np.array([
        int(g.y_bin.item())
        for g in graph_list
    ])

    n_total = len(y)

    n_benign = int(
        np.sum(y == 0)
    )

    n_attack = int(
        np.sum(y == 1)
    )

    benign_pct = (
        100.0 * n_benign / n_total
    )

    attack_pct = (
        100.0 * n_attack / n_total
    )

    return {
        "split": split_name,
        "total_windows": n_total,
        "benign_windows": n_benign,
        "benign_%": benign_pct,
        "attack_windows": n_attack,
        "attack_%": attack_pct
    }


composition_results = [
    binary_composition(
        train_graphs,
        "Train"
    ),
    binary_composition(
        val_graphs,
        "Validation"
    ),
    binary_composition(
        stream_graphs,
        "Streaming"
    )
]


df_composition = pd.DataFrame(
    composition_results
)


print("\nWindow-level composition:\n")

print(
    df_composition.to_string(
        index=False,
        formatters={
            "benign_%":
                lambda x: f"{x:.2f}",
            "attack_%":
                lambda x: f"{x:.2f}"
        }
    )
)


# =====================================================================
# 4. VERIFY STREAM COMPOSITION DIRECTLY FROM PREDICTION FILE
# =====================================================================

y_true = df_pred[
    "bin_true"
].to_numpy().astype(int)

attack_prob = df_pred[
    "attack_prob"
].to_numpy().astype(float)

n_total = len(y_true)

n_benign = int(
    np.sum(y_true == 0)
)

n_attack = int(
    np.sum(y_true == 1)
)

print("\nStreaming composition from saved predictions:")

print(
    f"Total windows : {n_total:,}"
)

print(
    f"Benign        : {n_benign:,} "
    f"({100*n_benign/n_total:.2f}%)"
)

print(
    f"Attack        : {n_attack:,} "
    f"({100*n_attack/n_total:.2f}%)"
)


# =====================================================================
# 5. ORIGINAL ARGMAX RESULTS
#
# This reproduces the seed-2026 result using the predictions made
# directly by model.argmax().
# =====================================================================

print("\n" + "=" * 100)
print("B. ORIGINAL ARGMAX BINARY RESULT")
print("=" * 100)

y_pred_argmax = df_pred[
    "bin_pred"
].to_numpy().astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true,
    y_pred_argmax,
    labels=[0, 1]
).ravel()

argmax_accuracy = accuracy_score(
    y_true,
    y_pred_argmax
)

argmax_precision = precision_score(
    y_true,
    y_pred_argmax,
    pos_label=1,
    zero_division=0
)

argmax_recall = recall_score(
    y_true,
    y_pred_argmax,
    pos_label=1,
    zero_division=0
)

argmax_f1 = f1_score(
    y_true,
    y_pred_argmax,
    pos_label=1,
    zero_division=0
)

argmax_fnr = (
    fn / (fn + tp)
)

argmax_fpr = (
    fp / (fp + tn)
)


print(
    f"Accuracy        : {argmax_accuracy:.4f}"
)

print(
    f"Attack precision: {argmax_precision:.4f}"
)

print(
    f"Attack recall   : {argmax_recall:.4f}"
)

print(
    f"Attack F1       : {argmax_f1:.4f}"
)

print(
    f"FNR             : {argmax_fnr:.4f}"
)

print(
    f"Benign FPR      : {argmax_fpr:.4f} "
    f"({100*argmax_fpr:.2f}%)"
)

print(
    f"\nTN={tn:,}, FP={fp:,}, "
    f"FN={fn:,}, TP={tp:,}"
)


# =====================================================================
# 6. THRESHOLD SENSITIVITY
#
# Attack if:
#       P(attack) >= threshold
#
# This is the correct variable for threshold analysis.
# =====================================================================

print("\n" + "=" * 100)
print("C. BINARY THRESHOLD SENSITIVITY")
print("=" * 100)


thresholds = np.arange(
    0.50,
    0.951,
    0.05
)

threshold_results = []


for threshold in thresholds:

    y_pred_thr = (
        attack_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred_thr,
        labels=[0, 1]
    ).ravel()


    accuracy = accuracy_score(
        y_true,
        y_pred_thr
    )

    precision = precision_score(
        y_true,
        y_pred_thr,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred_thr,
        pos_label=1,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred_thr,
        pos_label=1,
        zero_division=0
    )


    fnr = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )

    benign_fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )


    threshold_results.append({

        "threshold":
            threshold,

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fnr":
            fnr,

        "benign_fpr":
            benign_fpr,

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    })


df_threshold = pd.DataFrame(
    threshold_results
)


print("\nThreshold sensitivity:\n")

print(
    df_threshold.to_string(
        index=False,
        formatters={
            "threshold":
                lambda x: f"{x:.2f}",
            "accuracy":
                lambda x: f"{x:.4f}",
            "precision":
                lambda x: f"{x:.4f}",
            "recall":
                lambda x: f"{x:.4f}",
            "f1":
                lambda x: f"{x:.4f}",
            "fnr":
                lambda x: f"{x:.4f}",
            "benign_fpr":
                lambda x: f"{x:.4f}"
        }
    )
)


# =====================================================================
# 7. FIND THRESHOLD WITH BEST F1
# =====================================================================

best_idx = df_threshold[
    "f1"
].idxmax()

best_row = df_threshold.loc[
    best_idx
]

print("\n" + "-" * 70)

print(
    "Best threshold according to attack F1:"
)

print(
    f"Threshold  = {best_row['threshold']:.2f}"
)

print(
    f"Precision  = {best_row['precision']:.4f}"
)

print(
    f"Recall     = {best_row['recall']:.4f}"
)

print(
    f"F1         = {best_row['f1']:.4f}"
)

print(
    f"Benign FPR = "
    f"{best_row['benign_fpr']:.4f} "
    f"({100*best_row['benign_fpr']:.2f}%)"
)

print(
    f"FNR        = {best_row['fnr']:.4f}"
)


# =====================================================================
# 8. PRECISION–RECALL CURVE
# =====================================================================

print("\n" + "=" * 100)
print("D. PRECISION–RECALL ANALYSIS")
print("=" * 100)


pr_precision, pr_recall, pr_thresholds = \
    precision_recall_curve(
        y_true,
        attack_prob,
        pos_label=1
    )


average_precision = \
    average_precision_score(
        y_true,
        attack_prob
    )


print(
    f"Average Precision / PR-AUC = "
    f"{average_precision:.6f}"
)


# =====================================================================
# 9. PRECISION–RECALL FIGURE
# =====================================================================

plt.figure(
    figsize=(7, 6)
)

plt.plot(
    pr_recall,
    pr_precision,
    linewidth=2
)

plt.xlabel(
    "Recall",
    fontsize=12
)

plt.ylabel(
    "Precision",
    fontsize=12
)

plt.title(
    f"Precision–Recall Curve "
    f"(AP = {average_precision:.4f})",
    fontsize=13
)

plt.grid(
    alpha=0.3
)

plt.xlim(
    0,
    1.01
)

plt.ylim(
    0,
    1.01
)

plt.tight_layout()

plt.savefig(
    "binary_precision_recall_curve.pdf",
    bbox_inches="tight"
)

plt.savefig(
    "binary_precision_recall_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================
# 10. THRESHOLD VS BENIGN FPR
# =====================================================================

plt.figure(
    figsize=(7, 5)
)

plt.plot(
    df_threshold["threshold"],
    100 * df_threshold["benign_fpr"],
    marker="o"
)

plt.xlabel(
    "Attack Probability Threshold",
    fontsize=12
)

plt.ylabel(
    "Benign False Positive Rate (%)",
    fontsize=12
)

plt.title(
    "Threshold Sensitivity of Benign False Positive Rate",
    fontsize=13
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "threshold_vs_benign_fpr.pdf",
    bbox_inches="tight"
)

plt.savefig(
    "threshold_vs_benign_fpr.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================
# 11. THRESHOLD VS PRECISION / RECALL / F1
# =====================================================================

plt.figure(
    figsize=(7, 5)
)

plt.plot(
    df_threshold["threshold"],
    df_threshold["precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    df_threshold["threshold"],
    df_threshold["recall"],
    marker="s",
    label="Recall"
)

plt.plot(
    df_threshold["threshold"],
    df_threshold["f1"],
    marker="^",
    label="F1-score"
)

plt.xlabel(
    "Attack Probability Threshold",
    fontsize=12
)

plt.ylabel(
    "Score",
    fontsize=12
)

plt.title(
    "Binary Threshold Sensitivity",
    fontsize=13
)

plt.ylim(
    0,
    1.02
)

plt.grid(
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "binary_threshold_sensitivity.pdf",
    bbox_inches="tight"
)

plt.savefig(
    "binary_threshold_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================
# 12. SAVE ALL NUMERICAL RESULTS
# =====================================================================

df_composition.to_csv(
    "window_level_class_composition.csv",
    index=False
)

df_threshold.to_csv(
    "binary_threshold_sensitivity.csv",
    index=False
)


pr_df = pd.DataFrame({
    "recall":
        pr_recall,

    "precision":
        pr_precision
})

pr_df.to_csv(
    "binary_precision_recall_curve.csv",
    index=False
)


summary_operating_point = pd.DataFrame([{

    "stream_windows":
        len(y_true),

    "benign_windows":
        n_benign,

    "benign_percentage":
        100 * n_benign / len(y_true),

    "attack_windows":
        n_attack,

    "attack_percentage":
        100 * n_attack / len(y_true),

    "argmax_benign_fpr":
        argmax_fpr,

    "argmax_fnr":
        argmax_fnr,

    "argmax_precision":
        argmax_precision,

    "argmax_recall":
        argmax_recall,

    "argmax_f1":
        argmax_f1,

    "average_precision":
        average_precision,

    "best_f1_threshold":
        best_row["threshold"],

    "best_threshold_f1":
        best_row["f1"],

    "best_threshold_benign_fpr":
        best_row["benign_fpr"]
}])


summary_operating_point.to_csv(
    "operating_point_summary.csv",
    index=False
)


print("\n" + "=" * 100)
print("FILES SAVED")
print("=" * 100)

print("""
1. window_level_class_composition.csv
2. binary_threshold_sensitivity.csv
3. binary_precision_recall_curve.csv
4. operating_point_summary.csv
5. binary_precision_recall_curve.pdf
6. binary_precision_recall_curve.png
7. threshold_vs_benign_fpr.pdf
8. threshold_vs_benign_fpr.png
9. binary_threshold_sensitivity.pdf
10. binary_threshold_sensitivity.png
""")

In [ ]:
# ============================================================
# OPERATING-POINT FIGURES (Figure 4)
# ============================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    1, 3,
    figsize=(10, 3.4)
)

# ------------------------------------------------------------
# (a) Precision–Recall curve
# ------------------------------------------------------------

axes[0].plot(
    pr_recall,
    pr_precision,
    linewidth=2,
    label=f"AP = {average_precision:.4f}"
)

axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("(a) Precision–Recall Curve")
axes[0].set_xlim(0, 1.01)
axes[0].set_ylim(0, 1.01)
axes[0].grid(alpha=0.3)
axes[0].legend(loc="lower left")


# ------------------------------------------------------------
# (b) Threshold sensitivity
# ------------------------------------------------------------

axes[1].plot(
    df_threshold["threshold"],
    df_threshold["precision"],
    marker="o",
    label="Precision"
)

axes[1].plot(
    df_threshold["threshold"],
    df_threshold["recall"],
    marker="s",
    label="Recall"
)

axes[1].plot(
    df_threshold["threshold"],
    df_threshold["f1"],
    marker="^",
    label="F1-score"
)

axes[1].set_xlabel("Attack Probability Threshold")
axes[1].set_ylabel("Score")
axes[1].set_title("(b) Threshold Sensitivity")
axes[1].set_ylim(0.94, 1.002)
axes[1].grid(alpha=0.3)
axes[1].legend(loc="lower left")


# ------------------------------------------------------------
# (c) Benign FPR and FNR
# ------------------------------------------------------------

axes[2].plot(
    df_threshold["threshold"],
    100 * df_threshold["benign_fpr"],
    marker="o",
    label="Benign FPR"
)

axes[2].plot(
    df_threshold["threshold"],
    100 * df_threshold["fnr"],
    marker="s",
    label="FNR"
)

axes[2].set_xlabel("Attack Probability Threshold")
axes[2].set_ylabel("Rate (%)")
axes[2].set_title("(c) False Positive / Negative Trade-off")
axes[2].grid(alpha=0.3)
axes[2].legend()


# ------------------------------------------------------------
# Layout + save
# ------------------------------------------------------------

plt.tight_layout()

plt.savefig(
    "operating_point_combined_analysis.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "operating_point_combined_analysis.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.title("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history["train_acc"], label="train")
plt.plot(history["val_acc"], label="val")
plt.title("Accuracy")
plt.legend()

plt.show()

In [ ]:
def warmup_memory(self, train_graphs, batch_size=32):
        """
        One chronological pass through training data to populate
        TGN memory before streaming begins.
        """
        print('Warming up TGN memory (no gradients)...')
        self.tgn_memory.reset()
        loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=False)
        self.model.eval()
        
        with torch.no_grad():
            for data in loader:
                data = data.to(self.device)
                _, _, _, node_emb, nids = self.model(data, self.tgn_memory)
                if nids is not None:
                    self.tgn_memory.update_with_gru(
                        nids, node_emb.detach(), # Ensure detach for memory safety
                        self.model.memory_updater, self.device)
        
        print(f'Memory warm-up done. Current memory usage: {self.tgn_memory.stats()}')

        # Update drift reference if detector exists
        if self.drift_detector is not None:
            emb_list = []
            with torch.no_grad():
                for data in loader:
                    data = datza.to(self.device)
                    emb_list.append(self.model.get_graph_embedding(data, self.tgn_memory).cpu())
            all_emb = torch.cat(emb_list, dim=0)
            self.drift_detector.update_reference(all_emb.numpy())
            print('Drift reference updated.')

In [ ]:
import torch
from torch_geometric.loader import DataLoader

def warmup_memory_patched(self, train_graphs, batch_size=16):
    """
    One chronological pass through training data to populate
    TGN memory before streaming begins.
    """
    print(f'Warming up TGN memory for {len(train_graphs)} graphs...')
    self.tgn_memory.reset()
    
    # We use a DataLoader to handle the graphs in order
    loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=False)
    self.model.eval()
    
    with torch.no_grad():
        for data in loader:
            data = data.to(self.device)
            # Forward pass to get embeddings and node IDs
            _, _, _, node_emb, nids = self.model(data, self.tgn_memory)
            
            if nids is not None:
                # Update the temporal memory without calculating gradients
                self.tgn_memory.update_with_gru(
                    nids, 
                    node_emb.detach(), 
                    self.model.memory_updater, 
                    self.device
                )
    
    print('Memory warm-up complete.')

# This line "plugs" the function into your existing trainer object
import types
trainer.warmup_memory = types.MethodType(warmup_memory_patched, trainer)

print("✅ warmup_memory method has been successfully added to the trainer.")

In [ ]:
# =============================================================================
# PHASE 2: TGN MEMORY WARM-UP & EWC SNAPSHOT (STRATIFIED/CPU VERSION)
# =============================================================================

# 1. Restore the Best Weights from Phase 1
# This ensures that if Epoch 39 was 85% and Epoch 41 was 84%, 
# we use the 85% version for the warmup and EWC.
print("Restoring best model weights from Initial Training...")
# If your early stopping object is named 'early', we restore from there:
# early.restore(model, device) 

# 2. Populate the TGN "Brain"
# We use the full training set (58k graphs) to ensure every IP's 
# baseline behavior is recorded in the tgn_memory.
print("Starting TGN Memory Warm-up...")
trainer.warmup_memory(train_graphs, batch_size=16)

# 3. Snapshot EWC (Elastic Weight Consolidation)
# Since we have 10 classes, we use a stratified subset for EWC 
# to ensure rare classes like Ransomware are represented in the Fisher Matrix.
print("Computing EWC Fisher Information Matrix on stratified subset...")
# Using the first 2000 is okay because your train_graphs are now stratified
ewc_loader = DataLoader(train_graphs[:2000], batch_size=16, shuffle=False)

# lambda_ewc=150 is a 'strong' penalty, perfect for keeping 
# the 85% accuracy stable during the streaming phase.
trainer.ewc = EWC(model, ewc_loader, device, lambda_ewc=150)

print('\n' + '='*60)
print('✅ PHASE 2 COMPLETE')
print(f'Memory State: {len(trainer.tgn_memory.memory)} unique nodes initialized.')
print('Drift Reference: Set to Stratified Training Distribution.')
print('EWC: Active (Protecting 85%+ Multiclass Logic).')
print('='*60)

In [ ]:
# Initialize history lists in the trainer if they don't exist
trainer.train_losses, trainer.val_losses = [], []
trainer.train_bin_accs, trainer.val_bin_accs = [], []
trainer.train_multi_accs, trainer.val_multi_accs = [], []

# Note: You should have run the training loop with code that appends to these.
# If you already finished training and didn't save them, 
# you'll need to manually populate them from your printed logs or re-run.

In [ ]:
import time
import numpy as np
import torch.nn.functional as F
from torch_geometric.data import Batch
UPDATE_INTERVAL = 32   # or 64 depending on your CPU budget
# =============================================================================
# PHASE 3: STREAMING SIMULATION WITH TGN MEMORY + ONLINE ADAPTATION
# =============================================================================

# Configuration
CONFIDENCE_THRESHOLD = 0.85

stream_results = []
accumulated    = []
latencies      = []
alerts         = []

model.eval()
# Ensure TGN memory is NOT reset here (we use the Warm-up state from Phase 2)
print('='*60)
print(f'STARTING STREAMING SIMULATION (Device: {device})')
print(f'Total Windows to Process: {len(stream_graphs)}')
print('='*60)

for i, graph in enumerate(stream_graphs):
    t0 = time.perf_counter()

    # ---- 1. INFERENCE (Live Detection) ----
    with torch.no_grad():
        # Move graph to CPU/Device
        g_batch = Batch.from_data_list([graph]).to(device)
        
        # Forward pass using the current TGN memory state
        bin_l, multi_l, _, node_emb, nids = model(g_batch, tgn_memory)

        # Calculate Probabilities
        bin_probs   = F.softmax(bin_l,   dim=1)
        multi_probs = F.softmax(multi_l, dim=1)

        # Extract Predictions & Confidence
        bin_pred    = int(bin_l.argmax(1).item())
        bin_conf    = float(bin_probs.max().item())
        multi_pred  = int(multi_l.argmax(1).item())
        multi_conf  = float(multi_probs.max().item())

    # ---- 2. LIVE TGN MEMORY UPDATE ----
    # This keeps the model "aware" of recent IP behavior even without training
    if nids is not None:
        tgn_memory.update_with_gru(
            nids, node_emb.detach(), model.memory_updater, device)

    # Track Latency (Crucial for real-time IDS)
    lat_ms = (time.perf_counter() - t0) * 1000
    latencies.append(lat_ms)

    # ---- 3. GROUND TRUTH & ALERTS ----
    true_bin   = int(graph.y_bin.item())
    true_multi = int(graph.y_multi.item())
    wid        = int(graph.window_id.item())

    # Map IDs to Attack Names
    type_pred_str = id2attack.get(multi_pred, '?') if bin_pred == 1 else 'benign'
    type_true_str = id2attack.get(true_multi, '?')

    # Alert Logic: Only trigger for high-confidence attacks
    if bin_pred == 1 and bin_conf >= CONFIDENCE_THRESHOLD:
        alerts.append({
            'window_id': wid, 'type': type_pred_str, 
            'conf': bin_conf, 'true': type_true_str
        })
        # Print first 10 alerts for visual confirmation
        if len(alerts) <= 10:
            print(f'🚨 ALERT [Win {wid:6d}] | PRED: {type_pred_str:<12} '
                  f'CONF: {bin_conf:.3f} | TRUE: {type_true_str}')

    stream_results.append({
        'window_id' : wid,
        'bin_pred'  : bin_pred,
        'bin_conf'  : bin_conf,
        'bin_true'  : true_bin,
        'multi_pred': multi_pred,
        'multi_conf': multi_conf,
        'multi_true': true_multi,
        'type_pred' : type_pred_str,
        'type_true' : type_true_str,
        'latency_ms': lat_ms,
    })

    # ---- 4. ONLINE ADAPTATION (Learning while Streaming) ----
    # We accumulate a small batch of traffic to 'fine-tune' the model
    accumulated.append(graph)
    if len(accumulated) >= UPDATE_INTERVAL:
        # Patch check: ensuring the trainer has incremental_update
        if hasattr(trainer, 'incremental_update'):
            trainer.incremental_update(accumulated, n_steps=1) # 1 step for CPU speed
        
        # Switch back to eval mode after update
        model.eval()
        accumulated = []

    # ---- 5. PROGRESS REPORTING ----
    if (i+1) % 500 == 0:
        recent = stream_results[-500:]
        acc_r = sum(1 for r in recent if r['bin_pred'] == r['bin_true']) / 500
        avg_lat = np.mean(latencies[-500:])
        print(f'📈 Progress: [{i+1:6d}/{len(stream_graphs)}] '
              f'Acc (last 500): {acc_r:.2f} | Latency: {avg_lat:.2f}ms | '
              f'Memory: {len(tgn_memory.memory)} IPs')

# Final Cleanup for any leftover graphs
if accumulated and hasattr(trainer, 'incremental_update'):
    trainer.incremental_update(accumulated, n_steps=1)

print('\n' + '='*60)
print('✅ STREAMING COMPLETE')
print(f'Total processed    : {len(stream_results)}')
print(f'Total alerts       : {len(alerts)}')
print(f'Average Latency    : {np.mean(latencies):.2f} ms')
print(f'95th Percentile Lat: {np.percentile(latencies, 95):.2f} ms')
print('='*60)

In [ ]:
# =============================================================================
# BINARY DETECTION EVALUATION
# =============================================================================
y_true_bin = [r['bin_true'] for r in stream_results]
y_pred_bin = [r['bin_pred'] for r in stream_results]

print('='*55)
print('BINARY DETECTION RESULTS (streaming phase)')
print('='*55)
print(f'Total windows   : {len(y_true_bin)}')
print(f'Accuracy        : {accuracy_score(y_true_bin, y_pred_bin):.4f}')
print(f'Balanced Acc    : {balanced_accuracy_score(y_true_bin, y_pred_bin):.4f}')
print()
print(classification_report(
    y_true_bin, y_pred_bin,
    target_names=['Benign(0)', 'Attack(1)'],
    digits=4, zero_division=0))

cm_bin = confusion_matrix(y_true_bin, y_pred_bin)
tn, fp, fn, tp = cm_bin.ravel()
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')
print(f'False Negative Rate (missed attacks): {fn/(fn+tp):.4f}')

fig, ax = plt.subplots(figsize=(3.5, 3.0), dpi=200)
sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Benign','Pred Attack'],
            yticklabels=['True Benign','True Attack'], ax=ax)
ax.set_title('Binary Confusion Matrix (Stream)')
plt.tight_layout()
plt.savefig('binary_cm.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# =============================================================================
# ATTACK-TYPE MULTICLASS EVALUATION (Excluding Benign)
# =============================================================================
# 1. Filter results to only include windows where the GROUND TRUTH was an attack
attack_results = [r for r in stream_results if r['bin_true'] == 1]

y_true_type = [r['type_true'] for r in attack_results]
y_pred_type = [r['type_pred'] for r in attack_results]

# 2. Define the labels to include (Excluding 'benign')
# We take all unique types found in the data, then remove 'benign' if present
labels = sorted(list(set(y_true_type) | set(y_pred_type)))
if 'benign' in labels:
    labels.remove('benign')

print('='*55)
print('🔥 ATTACK-TYPE FORENSICS (True-Attack Windows Only)')
print('='*55)
print(f'Total attack windows analyzed: {len(y_true_type)}')
print(f'Classification Accuracy      : {accuracy_score(y_true_type, y_pred_type):.4f}')
print('-'*55)

# 3. Print report using only the filtered labels
print(classification_report(
    y_true_type, y_pred_type,
    labels=labels,        # This forces the report to only show these classes
    digits=4, 
    zero_division=0))

# 4. Generate Confusion Matrix without the benign row/column
cm_multi = confusion_matrix(y_true_type, y_pred_type, labels=labels)

fig, ax = plt.subplots(figsize=(8, 6), dpi=200)
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax,
            cbar_kws={'label': 'Number of Instances'})

ax.set_title('Attack-Type Confusion Matrix (Streaming Phase)')
ax.set_xlabel('Predicted Attack Type')
ax.set_ylabel('True Attack Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('multiclass_attack_only_cm.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# =============================================================================
# PER-CLASS RECALL BAR CHART (key diagnostic plot for the paper)
# =============================================================================
from sklearn.metrics import recall_score

classes_present = sorted(list(set(y_true_type)))
recalls = []
for c in classes_present:
    y_t = [1 if t == c else 0 for t in y_true_type]
    y_p = [1 if p == c else 0 for p in y_pred_type]
    r = recall_score(y_t, y_p, zero_division=0)
    recalls.append(r)

colors = ['#2ecc71' if r >= 0.70 else ('#f39c12' if r >= 0.40 else '#e74c3c')
          for r in recalls]

fig, ax = plt.subplots(figsize=(5.5, 2.8), dpi=300)
bars = ax.bar(classes_present, recalls, color=colors)
ax.set_ylim(0, 1.05)
ax.axhline(0.70, color='gray', linestyle='--', linewidth=0.8, label='Target 0.70')
ax.set_ylabel('Recall')
ax.set_title('Per-class Recall — Streaming Phase')
plt.xticks(rotation=40, ha='right', fontsize=8)
for bar, r in zip(bars, recalls):
    ax.text(bar.get_x()+bar.get_width()/2, r+0.02,
            f'{r:.2f}', ha='center', fontsize=7)
ax.legend(frameon=False, fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('per_class_recall.pdf', bbox_inches='tight')
plt.show()


In [ ]:
WINDOW = 10  # smaller window

window_acc = []
for i in range(0, len(stream_results), WINDOW):
    batch = stream_results[i:i+WINDOW]
    acc = sum(1 for r in batch if r['bin_pred']==r['bin_true']) / len(batch)
    window_acc.append(acc)

import pandas as pd
rolling_trend = pd.Series(window_acc).rolling(window=5, min_periods=1).mean()

# cumulative accuracy
cumulative_acc = []
correct = 0
for i, r in enumerate(stream_results, 1):
    if r['bin_pred'] == r['bin_true']:
        correct += 1
    cumulative_acc.append(correct / i)

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8), dpi=200)

# -------- Accuracy Plot --------
ax = axes[0]
ax.plot(window_acc, linewidth=1.0, alpha=0.5, label="Raw")
ax.plot(rolling_trend, linewidth=2.0, label="Trend")
ax.plot(cumulative_acc[::WINDOW], linestyle=':', linewidth=1.5, label="Cumulative")

ax.axhline(np.mean(window_acc), linestyle='--',
           linewidth=0.8, label=f'Mean={np.mean(window_acc):.3f}')

ax.set_xlabel(f'Batch of {WINDOW}')
ax.set_ylabel('Binary Accuracy')
ax.set_title('Streaming Accuracy Over Time')
ax.set_ylim(0, 1)
ax.legend(frameon=False)

# -------- Latency Plot --------
ax = axes[1]
ax.hist(latencies, bins=40, alpha=0.7)

ax.axvline(np.mean(latencies), linestyle='--',
           label=f'Mean={np.mean(latencies):.1f}ms')
ax.axvline(np.percentile(latencies,95), linestyle='--',
           label=f'P95={np.percentile(latencies,95):.1f}ms')

ax.set_xlabel('Inference Latency (ms)')
ax.set_ylabel('Count')
ax.set_title('Inference Latency Distribution')
ax.legend(frameon=False, fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Data Processing ---
WINDOW = 10  
window_acc = []
for i in range(0, len(stream_results), WINDOW):
    batch = stream_results[i:i+WINDOW]
    acc = sum(1 for r in batch if r['bin_pred']==r['bin_true']) / len(batch)
    window_acc.append(acc)

# Increase rolling window to 20 for a smoother trend line
rolling_trend = pd.Series(window_acc).rolling(window=20, min_periods=1).mean()

# Calculate cumulative accuracy
cumulative_acc = []
correct = 0
for i, r in enumerate(stream_results, 1):
    if r['bin_pred'] == r['bin_true']:
        correct += 1
    cumulative_acc.append(correct / i)

# --- 2. Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=120)

# -------- Enhanced Accuracy Plot --------
ax = axes[0]
ax.plot(window_acc, color='skyblue', linewidth=0.8, alpha=0.4, label="Raw Batch")
ax.plot(rolling_trend, color='darkorange', linewidth=2, label="Moving Average (20)")
ax.plot(cumulative_acc[::WINDOW], color='green', linestyle=':', linewidth=1.5, label="Cumulative")

mean_val = np.mean(window_acc)
ax.axhline(mean_val, color='steelblue', linestyle='--', linewidth=1, label=f'Mean={mean_val:.3f}')

ax.set_xlabel(f'Batch Index (Size {WINDOW})')
ax.set_ylabel('Binary Accuracy')
ax.set_title('Streaming Accuracy (Zoomed)')

# CHANGE: Zooming in on the relevant range to see fluctuations
ax.set_ylim(0.75, 1.02) 
ax.grid(axis='y', alpha=0.3)
ax.legend(frameon=True, loc='lower left', fontsize=8)

# -------- Enhanced Latency Plot --------
ax = axes[1]
# CHANGE: Log scale helps see the distribution when there are far-outliers
ax.hist(latencies, bins=50, color='royalblue', alpha=0.7, edgecolor='white', log=True)

mean_lat = np.mean(latencies)
p95_lat = np.percentile(latencies, 95)

ax.axvline(mean_lat, color='red', linestyle='--', label=f'Mean: {mean_lat:.1f}ms')
ax.axvline(p95_lat, color='black', linestyle='--', label=f'P95: {p95_lat:.1f}ms')

ax.set_xlabel('Inference Latency (ms)')
ax.set_ylabel('Count (Log Scale)')
ax.set_title('Inference Latency Distribution')
ax.legend(frameon=True, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Manually fill the buffer with the streaming graphs to visualize coverage
for graph in stream_graphs:
    replay_buffer.add(graph)

print(f"✅ Replay Buffer populated with {len(replay_buffer)} streaming samples.")
# Now you can re-run your bar chart code!

In [ ]:
import matplotlib.pyplot as plt

# =============================================================================
# STREAMING ANALYSIS: CONFIDENCE & TGN STATS
# =============================================================================
plt.rcParams.update({'font.family':'serif','font.size':9, 'axes.linewidth':0.8,'pdf.fonttype':42})

# We'll create a 1x2 plot: Binary Confidence and Multiclass Confidence
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), dpi=200)

# 1. Binary Confidence (How well it separates Benign vs Attack)
conf_atk = [r['bin_conf'] for r in stream_results if r['bin_true']==1]
conf_ben = [r['bin_conf'] for r in stream_results if r['bin_true']==0]

ax0 = axes[0]
ax0.hist(conf_atk, bins=25, alpha=0.6, label='Attack', color='crimson', edgecolor='white')
ax0.hist(conf_ben, bins=25, alpha=0.6, label='Benign', color='steelblue', edgecolor='white')
ax0.set_title('Detection Confidence Distribution')
ax0.set_xlabel('Confidence Score')
ax0.set_ylabel('Frequency')
ax0.legend(frameon=False)

# 2. Multiclass Performance (How well it identifies the 10 types)
# We plot the confidence of CORRECT predictions vs INCORRECT ones
conf_corr = [r['multi_conf'] for r in stream_results if r['multi_pred']==r['multi_true']]
conf_err  = [r['multi_conf'] for r in stream_results if r['multi_pred']!=r['multi_true']]

ax1 = axes[1]
if conf_corr: # Handle case where no samples exist yet
    ax1.hist(conf_corr, bins=25, alpha=0.6, label='Correct Class', color='forestgreen', edgecolor='white')
    ax1.hist(conf_err,  bins=25, alpha=0.6, label='Wrong Class', color='darkorange', edgecolor='white')
ax1.set_title('Classification Confidence (10-Class)')
ax1.set_xlabel('Confidence Score')
ax1.set_ylabel('Frequency')
ax1.legend(frameon=False)

# Styling clean-up
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('confidence_analysis.pdf', bbox_inches='tight')
plt.show()

# --- Print Detailed Stats ---
print('='*40)
print(f'TGN NODES TRACKED: {len(tgn_memory.memory)}')
print(f'AVG ATTACK CONFIDENCE: {np.mean(conf_atk):.4f}' if conf_atk else "N/A")
print(f'AVG BENIGN CONFIDENCE: {np.mean(conf_ben):.4f}' if conf_ben else "N/A")
print('='*40)

# Check if Replay Buffer is still empty
if len(replay_buffer) == 0:
    print("💡 NOTE: Replay Buffer is empty. This is normal if 'incremental_update' was")
    print("   called without a push_to_buffer step during the streaming loop.")

In [ ]:
# =============================================================================
# SAMPLE GRAPH VISUALIZATIONS (paper-quality)
# =============================================================================
from torch_geometric.utils import to_networkx

def plot_graph_paper(g_data, title, ax):
    G = to_networkx(g_data, to_undirected=False)
    pos = nx.spring_layout(G, seed=42)
    # Color nodes by attack participation if y_node available
    if hasattr(g_data, 'y_node'):
        node_colors = ['#e74c3c' if int(g_data.y_node[i].item()) == 1
                       else '#2980b9'
                       for i in range(g_data.num_nodes)]
    else:
        node_colors = '#2980b9'
    nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                           node_size=60, ax=ax)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle='-|>',
                           arrowsize=8, alpha=0.3, ax=ax)
    ax.set_title(title, fontsize=8)
    ax.axis('off')


benign_g = next((g for g in stream_graphs[:500] if g.y_bin.item()==0), None)
attack_g = next((g for g in stream_graphs[:500] if g.y_bin.item()==1), None)

fig, axes = plt.subplots(1, 2, figsize=(6, 3), dpi=300)

if benign_g is not None:
    plot_graph_paper(benign_g,
        f'Benign window  (nodes={benign_g.num_nodes})', axes[0])
if attack_g is not None:
    attack_name = id2attack[int(attack_g.y_multi.item())]
    plot_graph_paper(attack_g,
        f'{attack_name} window (nodes={attack_g.num_nodes})', axes[1])

# Legend
import matplotlib.patches as mpatches
red_p  = mpatches.Patch(color='#e74c3c', label='Attack node')
blue_p = mpatches.Patch(color='#2980b9', label='Benign node')
fig.legend(handles=[blue_p, red_p], loc='lower center',
           ncol=2, frameon=False, fontsize=7)

plt.suptitle('Sample Graph Snapshots (red = attack-participating node)',
             fontsize=8)
plt.tight_layout(rect=[0,0.08,1,1])
plt.savefig('sample_graphs.pdf', bbox_inches='tight')
plt.show()
print('All evaluations and visualizations completed.')


In [ ]:
from sklearn.metrics import confusion_matrix, recall_score

print('=' * 55)
print('PER-CLASS TRUE POSITIVE RATE (TPR) & FALSE POSITIVE RATE (FPR)')
print('=' * 55)

# 'labels' comes from the previous analysis (cell 88182a01)
# and contains only the attack class names (excluding 'benign').

tpr_per_class = {}
fpr_per_class = {}

for cls in labels:
    # Create binary true and predicted labels for the current class (one-vs-rest)
    y_true_binary_cls = [1 if t == cls else 0 for t in y_true_type]
    y_pred_binary_cls = [1 if p == cls else 0 for p in y_pred_type]

    # Calculate confusion matrix for the current class
    # cm = [[TN, FP], [FN, TP]]
    cm = confusion_matrix(y_true_binary_cls, y_pred_binary_cls, labels=[0, 1])

    TN, FP, FN, TP = cm.ravel() if cm.size == 4 else (0, 0, 0, 0) # Handle cases where a class might not appear

    # True Positive Rate (Recall)
    tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
    tpr_per_class[cls] = tpr

    # False Positive Rate
    fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
    fpr_per_class[cls] = fpr

    print(f"Class: {cls:<15} | TPR (Recall): {tpr:.4f} | FPR: {fpr:.4f}")

print('=' * 55)

In [ ]:
import os
import torch

save_dir = os.path.join("results", "nf_ton_iot_v3")
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "tgn_joint_model.pt")

torch.save({
    "model_state_dict": trainer.model.state_dict(),
    "optimizer_state_dict": trainer.optimizer.state_dict(),
    "best_acc": trainer.best_acc,
}, model_path)

print(f"Model saved at: {model_path}")

In [ ]:
model_path = os.path.join(save_dir, "tgn_best_model.pt")

torch.save({
    "model_state_dict": trainer.best_state,
    "best_acc": trainer.best_acc
}, model_path)

print(f"Best model saved at: {model_path}")

In [ ]:
checkpoint = torch.load(model_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])

print("Model loaded successfully!")
print("Best accuracy was:", checkpoint.get("best_acc", "N/A"))

In [ ]:
# =====================================================================
# EXPERIMENT (Section 5.9):
# CONTRIBUTION OF INTEGER / PERSISTENT IP IDENTIFIERS
#
# Conditions:
#   1. stable_ip        : original IPv4 integer IDs
#   2. permuted_stable : arbitrary 1-to-1 IDs, persistent across windows
#   3. nonpersistent    : fresh IDs for every node occurrence/window
#
# IMPORTANT:
# - NO RETRAINING.
# - Same trained seed-2026 checkpoint is reloaded before every condition.
# - x, edge_index, edge_attr, labels and topology are unchanged.
# - Only node_ids used by TGN memory are changed.
# =====================================================================

import os
import copy
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    confusion_matrix
)


# =====================================================================
# 0. CONFIGURATION
# =====================================================================

SEED = 2026

CHECKPOINT = "TGN_JointIDS_seed_2026.pt"

UPDATE_INTERVAL = 32
WARMUP_BATCH = 16
EWC_GRAPHS = 2000
EWC_BATCH = 16

VARIANTS = [
    "stable_ip",
    "permuted_stable",
    "nonpersistent"
]

assert os.path.exists(CHECKPOINT), (
    f"Checkpoint not found: {CHECKPOINT}"
)

print("=" * 100)
print("IDENTIFIER ABLATION EXPERIMENT")
print("=" * 100)
print("Checkpoint :", CHECKPOINT)
print("Seed       :", SEED)
print("Device     :", device)
print("Variants   :", VARIANTS)


# =====================================================================
# 1. REPRODUCIBILITY
# =====================================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# =====================================================================
# 2. SAFE GRAPH COPY
# =====================================================================

def clone_graph(g):
    """
    Clone a PyG Data object so that changing node_ids does not modify
    the original train/validation/stream graphs.
    """
    return g.clone()


def clone_graph_list(graphs):
    return [
        clone_graph(g)
        for g in graphs
    ]


# =====================================================================
# 3. VERIFY NODE_IDS EXIST
# =====================================================================

for split_name, graph_list in [
    ("train", train_graphs),
    ("validation", val_graphs),
    ("stream", stream_graphs)
]:
    if not hasattr(graph_list[0], "node_ids"):
        raise ValueError(
            f"{split_name} graphs do not contain node_ids."
        )

print("\n✓ node_ids found in all graph splits.")


# =====================================================================
# 4. COLLECT ALL ORIGINAL NODE IDS
# =====================================================================

def collect_unique_node_ids(*graph_lists):

    ids = set()

    for graph_list in graph_lists:

        for g in graph_list:

            for nid in g.node_ids.cpu().tolist():
                ids.add(int(nid))

    return sorted(ids)


all_original_ids = collect_unique_node_ids(
    train_graphs,
    val_graphs,
    stream_graphs
)

print(
    f"Unique original persistent node IDs: "
    f"{len(all_original_ids):,}"
)


# =====================================================================
# 5. CREATE FIXED RANDOM BIJECTION
#
# Important:
# This changes the integer VALUES but preserves host identity.
#
# Example:
# original IP ID 3232235808 -> arbitrary ID 1000341
#
# Same host receives same new ID in every window.
# =====================================================================

rng = np.random.default_rng(SEED)

permuted_values = (
    rng.permutation(
        len(all_original_ids)
    )
    + 1_000_000
)

stable_permutation = {
    old_id: int(new_id)
    for old_id, new_id
    in zip(
        all_original_ids,
        permuted_values
    )
}

print("✓ Stable random one-to-one permutation created.")


# =====================================================================
# 6. APPLY IDENTIFIER POLICY
# =====================================================================

def prepare_variant_graphs(
    train_original,
    val_original,
    stream_original,
    variant
):
    """
    Returns copied graph lists under one node-ID policy.

    stable_ip:
        exact original node_ids

    permuted_stable:
        arbitrary fixed one-to-one remapping
        same host keeps same new ID everywhere

    nonpersistent:
        every node occurrence in every graph receives a new ID
        therefore no cross-window host memory can be retrieved
    """

    train_v = clone_graph_list(
        train_original
    )

    val_v = clone_graph_list(
        val_original
    )

    stream_v = clone_graph_list(
        stream_original
    )


    # -------------------------------------------------------------
    # ORIGINAL
    # -------------------------------------------------------------

    if variant == "stable_ip":

        return (
            train_v,
            val_v,
            stream_v
        )


    # -------------------------------------------------------------
    # PERMUTED BUT STILL PERSISTENT
    # -------------------------------------------------------------

    elif variant == "permuted_stable":

        for graph_list in [
            train_v,
            val_v,
            stream_v
        ]:

            for g in graph_list:

                original = (
                    g.node_ids
                    .cpu()
                    .tolist()
                )

                remapped = [
                    stable_permutation[
                        int(x)
                    ]
                    for x in original
                ]

                g.node_ids = torch.tensor(
                    remapped,
                    dtype=torch.long
                )

        return (
            train_v,
            val_v,
            stream_v
        )


    # -------------------------------------------------------------
    # NON-PERSISTENT IDENTIFIERS
    #
    # Global counter ensures NO identifier is reused across windows.
    # This preserves the same graph structure/features but prevents
    # the model from recognizing the same host through memory.
    # -------------------------------------------------------------

    elif variant == "nonpersistent":

        counter = 10_000_000

        for graph_list in [
            train_v,
            val_v,
            stream_v
        ]:

            for g in graph_list:

                n_nodes = int(
                    g.num_nodes
                )

                fresh_ids = torch.arange(
                    counter,
                    counter + n_nodes,
                    dtype=torch.long
                )

                g.node_ids = fresh_ids

                counter += n_nodes

        return (
            train_v,
            val_v,
            stream_v
        )


    else:

        raise ValueError(
            f"Unknown variant: {variant}"
        )


# =====================================================================
# 7. SANITY CHECK:
# IDENTIFIER ABLATION MUST NOT CHANGE INPUT FEATURES / LABELS
# =====================================================================

def verify_graph_integrity(
    original_graphs,
    changed_graphs,
    split_name,
    variant,
    n_check=100
):

    n = min(
        n_check,
        len(original_graphs)
    )

    for i in range(n):

        a = original_graphs[i]
        b = changed_graphs[i]

        assert torch.equal(
            a.x.cpu(),
            b.x.cpu()
        ), f"x changed: {variant}/{split_name}/{i}"

        assert torch.equal(
            a.edge_index.cpu(),
            b.edge_index.cpu()
        ), f"edge_index changed: {variant}/{split_name}/{i}"

        if (
            a.edge_attr is not None
            and b.edge_attr is not None
        ):

            assert torch.equal(
                a.edge_attr.cpu(),
                b.edge_attr.cpu()
            ), (
                f"edge_attr changed: "
                f"{variant}/{split_name}/{i}"
            )

        assert int(
            a.y_bin.item()
        ) == int(
            b.y_bin.item()
        )

        assert int(
            a.y_multi.item()
        ) == int(
            b.y_multi.item()
        )


# =====================================================================
# 8. CREATE FRESH MODEL + TRAINER
#
# Same model parameters as original notebook.
# =====================================================================

def create_fresh_system():

    set_seed(SEED)

    experiment_model = TGNJointIDS(
        in_dim=IN_DIM,
        hidden_dim=HIDDEN_DIM,
        num_classes=NUM_CLASSES,
        memory_dim=MEMORY_DIM,
        num_layers=NUM_LAYERS,
        num_heads=NUM_HEADS,
        dropout=DROPOUT,
        edge_dim=EDGE_DIM,
        sagpool_ratio=SAGPOOL_R
    ).to(device)


    checkpoint = torch.load(
        CHECKPOINT,
        map_location=device
    )

    experiment_model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )


    experiment_memory = TGNNodeMemory(
        memory_dim=MEMORY_DIM
    )


    experiment_trainer = JointTrainer(
        model=experiment_model,
        tgn_memory=experiment_memory,
        device=device,
        bin_weights=final_bin_weights.to(device),
        multi_weights=final_multi_weights.to(device),
        lr=1e-3
    )


    return (
        experiment_model,
        experiment_memory,
        experiment_trainer
    )


# =====================================================================
# 9. MEMORY WARM-UP
# =====================================================================

def warmup_memory(
    model,
    memory,
    train_variant
):

    memory.reset()

    loader = DataLoader(
        train_variant,
        batch_size=WARMUP_BATCH,
        shuffle=False
    )

    model.eval()

    with torch.no_grad():

        for batch_idx, data in enumerate(loader):

            data = data.to(device)

            (
                _,
                _,
                _,
                node_emb,
                nids
            ) = model(
                data,
                memory
            )

            if nids is not None:

                memory.update_with_gru(
                    nids,
                    node_emb.detach(),
                    model.memory_updater,
                    device
                )

            if (
                batch_idx + 1
            ) % 1000 == 0:

                print(
                    f"  Warm-up "
                    f"{batch_idx+1:,} batches | "
                    f"memory={len(memory):,}"
                )


# =====================================================================
# 10. PER-CLASS METRIC FUNCTION
# =====================================================================

def class_metrics(
    y_true,
    y_pred,
    class_id
):

    p, r, f, s = \
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[class_id],
            average=None,
            zero_division=0
        )

    return (
        float(p[0]),
        float(r[0]),
        float(f[0]),
        int(s[0])
    )


# =====================================================================
# 11. RUN ONE IDENTIFIER CONDITION
# =====================================================================

def evaluate_identifier_variant(
    variant,
    train_variant,
    val_variant,
    stream_variant
):

    print("\n")
    print("=" * 100)
    print(
        f"VARIANT: {variant.upper()}"
    )
    print("=" * 100)


    # -------------------------------------------------------------
    # Fresh identical model
    # -------------------------------------------------------------

    model_v, memory_v, trainer_v = \
        create_fresh_system()

    print(
        "✓ Seed-2026 trained weights restored."
    )


    # -------------------------------------------------------------
    # Warm-up temporal memory
    # -------------------------------------------------------------

    print("\nMemory warm-up...")

    warmup_memory(
        model_v,
        memory_v,
        train_variant
    )

    warmup_memory_keys = len(
        memory_v
    )

    print(
        f"✓ Warm-up memory keys: "
        f"{warmup_memory_keys:,}"
    )


    # -------------------------------------------------------------
    # EWC reference
    # -------------------------------------------------------------

    print("\nComputing EWC reference...")

    ewc_loader = DataLoader(
        train_variant[:EWC_GRAPHS],
        batch_size=EWC_BATCH,
        shuffle=False
    )

    trainer_v.ewc = EWC(
        model_v,
        ewc_loader,
        device,
        lambda_ewc=150
    )

    print("✓ EWC ready.")


    # -------------------------------------------------------------
    # Stream evaluation
    # -------------------------------------------------------------

    y_bin_true = []
    y_bin_pred = []

    y_multi_true = []
    y_multi_pred = []

    latencies = []

    accumulated = []

    model_v.eval()

    n_stream = len(
        stream_variant
    )


    for i, graph in enumerate(
        stream_variant
    ):

        t0 = time.perf_counter()

        with torch.no_grad():

            batch = Batch.from_data_list(
                [graph]
            ).to(device)

            (
                bin_logits,
                multi_logits,
                _,
                node_emb,
                nids
            ) = model_v(
                batch,
                memory_v
            )

            bin_pred = int(
                bin_logits.argmax(
                    dim=1
                ).item()
            )

            multi_pred = int(
                multi_logits.argmax(
                    dim=1
                ).item()
            )


        # ---------------------------------------------------------
        # Update TGN memory
        # ---------------------------------------------------------

        if nids is not None:

            memory_v.update_with_gru(
                nids,
                node_emb.detach(),
                model_v.memory_updater,
                device
            )


        latency = (
            time.perf_counter()
            - t0
        ) * 1000.0

        latencies.append(
            latency
        )


        y_bin_true.append(
            int(
                graph.y_bin.item()
            )
        )

        y_bin_pred.append(
            bin_pred
        )

        y_multi_true.append(
            int(
                graph.y_multi.item()
            )
        )

        y_multi_pred.append(
            multi_pred
        )


        # ---------------------------------------------------------
        # Same online update procedure as original experiment
        # ---------------------------------------------------------

        accumulated.append(
            graph
        )

        if len(
            accumulated
        ) >= UPDATE_INTERVAL:

            trainer_v.incremental_update(
                accumulated,
                n_steps=1
            )

            model_v.eval()

            accumulated = []


        if (
            i + 1
        ) % 5000 == 0:

            print(
                f"  Stream "
                f"{i+1:,}/{n_stream:,} | "
                f"memory={len(memory_v):,}"
            )


    # Note:
    # Original streaming code performs updates when 32 graphs
    # accumulate. We do not need an extra final partial update,
    # because all predictions have already been produced.


    # -------------------------------------------------------------
    # Convert arrays
    # -------------------------------------------------------------

    y_bin_true = np.array(
        y_bin_true
    )

    y_bin_pred = np.array(
        y_bin_pred
    )

    y_multi_true = np.array(
        y_multi_true
    )

    y_multi_pred = np.array(
        y_multi_pred
    )


    # -------------------------------------------------------------
    # Binary metrics
    # -------------------------------------------------------------

    binary_accuracy = accuracy_score(
        y_bin_true,
        y_bin_pred
    )

    binary_weighted_f1 = f1_score(
        y_bin_true,
        y_bin_pred,
        average="weighted",
        zero_division=0
    )

    attack_f1 = f1_score(
        y_bin_true,
        y_bin_pred,
        pos_label=1,
        zero_division=0
    )


    tn, fp, fn, tp = \
        confusion_matrix(
            y_bin_true,
            y_bin_pred,
            labels=[0, 1]
        ).ravel()


    benign_fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    fnr = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )


    # -------------------------------------------------------------
    # Attack-only multiclass metrics
    #
    # Ground-truth benign windows excluded.
    # Predictions of benign for attack windows remain errors.
    # -------------------------------------------------------------

    attack_mask = (
        y_bin_true == 1
    )

    yt = y_multi_true[
        attack_mask
    ]

    yp = y_multi_pred[
        attack_mask
    ]


    attack_ids = [
        attack2id[x]
        for x in ATTACK_CLASSES
        if x != "benign"
    ]


    multiclass_weighted_f1 = f1_score(
        yt,
        yp,
        labels=attack_ids,
        average="weighted",
        zero_division=0
    )


    multiclass_macro_f1 = f1_score(
        yt,
        yp,
        labels=attack_ids,
        average="macro",
        zero_division=0
    )


    # -------------------------------------------------------------
    # Scanning
    # -------------------------------------------------------------

    (
        scanning_precision,
        scanning_recall,
        scanning_f1,
        scanning_support
    ) = class_metrics(
        yt,
        yp,
        attack2id[
            "scanning"
        ]
    )


    # -------------------------------------------------------------
    # MITM
    # -------------------------------------------------------------

    (
        mitm_precision,
        mitm_recall,
        mitm_f1,
        mitm_support
    ) = class_metrics(
        yt,
        yp,
        attack2id[
            "mitm"
        ]
    )


    # -------------------------------------------------------------
    # Ransomware
    # -------------------------------------------------------------

    (
        ransomware_precision,
        ransomware_recall,
        ransomware_f1,
        ransomware_support
    ) = class_metrics(
        yt,
        yp,
        attack2id[
            "ransomware"
        ]
    )


    # -------------------------------------------------------------
    # Final result
    # -------------------------------------------------------------

    result = {

        "variant":
            variant,

        "binary_accuracy":
            binary_accuracy,

        "binary_weighted_f1":
            binary_weighted_f1,

        "attack_f1":
            attack_f1,

        "benign_fpr":
            benign_fpr,

        "fnr":
            fnr,

        "multiclass_weighted_f1":
            multiclass_weighted_f1,

        "multiclass_macro_f1":
            multiclass_macro_f1,

        "scanning_precision":
            scanning_precision,

        "scanning_recall":
            scanning_recall,

        "scanning_f1":
            scanning_f1,

        "scanning_support":
            scanning_support,

        "mitm_precision":
            mitm_precision,

        "mitm_recall":
            mitm_recall,

        "mitm_f1":
            mitm_f1,

        "mitm_support":
            mitm_support,

        "ransomware_precision":
            ransomware_precision,

        "ransomware_recall":
            ransomware_recall,

        "ransomware_f1":
            ransomware_f1,

        "ransomware_support":
            ransomware_support,

        "warmup_memory_keys":
            warmup_memory_keys,

        "final_memory_keys":
            len(memory_v),

        "mean_latency_ms":
            float(
                np.mean(
                    latencies
                )
            ),

        "p95_latency_ms":
            float(
                np.percentile(
                    latencies,
                    95
                )
            )
    }


    print("\nRESULTS")

    for k, v in result.items():

        if isinstance(
            v,
            (float, np.floating)
        ):

            print(
                f"{k:<30}: {v:.4f}"
            )

        else:

            print(
                f"{k:<30}: {v}"
            )


    return result


# =====================================================================
# 12. EXECUTE ALL THREE CONDITIONS
# =====================================================================

all_results = []


for variant in VARIANTS:

    print("\n")
    print("#" * 100)
    print(
        f"PREPARING {variant}"
    )
    print("#" * 100)


    train_v, val_v, stream_v = \
        prepare_variant_graphs(
            train_graphs,
            val_graphs,
            stream_graphs,
            variant
        )


    # Verify graph content did not change
    verify_graph_integrity(
        train_graphs,
        train_v,
        "train",
        variant
    )

    verify_graph_integrity(
        val_graphs,
        val_v,
        "validation",
        variant
    )

    verify_graph_integrity(
        stream_graphs,
        stream_v,
        "stream",
        variant
    )


    print(
        "✓ x, edges and labels unchanged."
    )


    # -------------------------------------------------------------
    # Demonstrate identifier behavior
    # -------------------------------------------------------------

    print(
        "\nExample original IDs :",
        train_graphs[0]
        .node_ids[:5]
        .cpu()
        .tolist()
    )

    print(
        "Variant IDs          :",
        train_v[0]
        .node_ids[:5]
        .cpu()
        .tolist()
    )


    result = \
        evaluate_identifier_variant(
            variant,
            train_v,
            val_v,
            stream_v
        )


    all_results.append(
        result
    )


    # Free copied graphs before next condition
    del train_v
    del val_v
    del stream_v

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =====================================================================
# 13. FINAL COMPARISON TABLE
# =====================================================================

df_id_ablation = pd.DataFrame(
    all_results
)


important_cols = [

    "variant",

    "binary_accuracy",
    "binary_weighted_f1",
    "benign_fpr",
    "fnr",

    "multiclass_weighted_f1",
    "multiclass_macro_f1",

    "scanning_precision",
    "scanning_recall",
    "scanning_f1",

    "mitm_recall",
    "mitm_f1",

    "ransomware_recall",
    "ransomware_f1",

    "warmup_memory_keys",
    "final_memory_keys"
]


print("\n")
print("=" * 120)
print("IDENTIFIER ABLATION — FINAL COMPARISON")
print("=" * 120)

print(
    df_id_ablation[
        important_cols
    ].to_string(
        index=False
    )
)


# =====================================================================
# 14. DIFFERENCE RELATIVE TO ORIGINAL STABLE IP
# =====================================================================

baseline = (
    df_id_ablation[
        df_id_ablation[
            "variant"
        ] == "stable_ip"
    ]
    .iloc[0]
)


delta_metrics = [
    "binary_accuracy",
    "multiclass_weighted_f1",
    "scanning_recall",
    "scanning_f1",
    "mitm_recall",
    "ransomware_recall",
    "benign_fpr"
]


delta_rows = []


for _, row in df_id_ablation.iterrows():

    d = {
        "variant":
            row[
                "variant"
            ]
    }

    for metric in delta_metrics:

        d[
            f"delta_{metric}"
        ] = (
            row[metric]
            - baseline[metric]
        )

    delta_rows.append(
        d
    )


df_deltas = pd.DataFrame(
    delta_rows
)


print("\n")
print("=" * 120)
print("CHANGE RELATIVE TO STABLE IP")
print("=" * 120)

print(
    df_deltas.to_string(
        index=False
    )
)


# =====================================================================
# 15. SAVE TABLES
# =====================================================================

df_id_ablation.to_csv(
    "identifier_ablation_results.csv",
    index=False
)

df_deltas.to_csv(
    "identifier_ablation_deltas.csv",
    index=False
)


print(
    "\n✓ Saved:"
    "\n  identifier_ablation_results.csv"
    "\n  identifier_ablation_deltas.csv"
)


# =====================================================================
# 16. PUBLICATION FIGURE
# =====================================================================

plot_df = (
    df_id_ablation
    .set_index(
        "variant"
    )
)


fig, axes = plt.subplots(
    1,
    3,
    figsize=(12, 3.8)
)


# -------------------------------------------------------------
# (a) Overall attack-type performance
# -------------------------------------------------------------

axes[0].bar(
    plot_df.index,
    plot_df[
        "multiclass_weighted_f1"
    ]
)

axes[0].set_ylabel(
    "Weighted F1"
)

axes[0].set_title(
    "(a) Overall Performance"
)

axes[0].set_ylim(
    0,
    1.0
)

axes[0].tick_params(
    axis="x",
    rotation=20
)

axes[0].grid(
    axis="y",
    alpha=0.3
)


# -------------------------------------------------------------
# (b) Scanning
# -------------------------------------------------------------

x = np.arange(
    len(
        plot_df.index
    )
)

width = 0.36

axes[1].bar(
    x - width/2,
    plot_df[
        "scanning_recall"
    ],
    width,
    label="Recall"
)

axes[1].bar(
    x + width/2,
    plot_df[
        "scanning_f1"
    ],
    width,
    label="F1-score"
)

axes[1].set_xticks(
    x
)

axes[1].set_xticklabels(
    plot_df.index,
    rotation=20
)

axes[1].set_ylim(
    0,
    1.0
)

axes[1].set_title(
    "(b) Scanning Performance"
)

axes[1].set_ylabel(
    "Score"
)

axes[1].legend()

axes[1].grid(
    axis="y",
    alpha=0.3
)


# -------------------------------------------------------------
# (c) Rare attack recall
# -------------------------------------------------------------

axes[2].bar(
    x - width/2,
    plot_df[
        "mitm_recall"
    ],
    width,
    label="MITM"
)

axes[2].bar(
    x + width/2,
    plot_df[
        "ransomware_recall"
    ],
    width,
    label="Ransomware"
)

axes[2].set_xticks(
    x
)

axes[2].set_xticklabels(
    plot_df.index,
    rotation=20
)

axes[2].set_ylim(
    0,
    1.0
)

axes[2].set_title(
    "(c) Rare-Class Recall"
)

axes[2].set_ylabel(
    "Recall"
)

axes[2].legend()

axes[2].grid(
    axis="y",
    alpha=0.3
)


plt.tight_layout()

plt.savefig(
    "identifier_ablation_combined.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "identifier_ablation_combined.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =====================================================================
# CORRECT 5-SEED REPETITION
# TGN-JointIDS
#
# Fixed split for all seeds
# Independent initialization for every seed
# Train -> save checkpoint -> warm-up -> EWC -> streaming evaluation
# =====================================================================

import os
import gc
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix
)


# =====================================================================
# 1. CONFIGURATION
# =====================================================================

SEEDS = [13, 42, 123, 777, 2026]

MAX_EPOCHS = 150

TRAIN_BATCH = 32
VAL_BATCH = 32

WARMUP_BATCH = 16
EWC_BATCH = 16
EWC_N_GRAPHS = 2000

UPDATE_INTERVAL = 32


# =====================================================================
# 2. DEVICE
# =====================================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 90)
print("5-SEED REPETITION")
print("=" * 90)

print(f"Device: {device}")

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# =====================================================================
# 3. FIXED SPLIT FOR ALL SEEDS
#
# IMPORTANT:
# train_graphs / val_graphs / stream_graphs must already have been
# generated ONCE before this cell.
#
# They are NOT regenerated inside the seed loop.
# =====================================================================

print("\nFixed split:")

print(
    f"Train      : {len(train_graphs):,}"
)

print(
    f"Validation : {len(val_graphs):,}"
)

print(
    f"Streaming  : {len(stream_graphs):,}"
)


train_loader = DataLoader(
    train_graphs,
    batch_size=TRAIN_BATCH,
    shuffle=False
)

val_loader = DataLoader(
    val_graphs,
    batch_size=VAL_BATCH,
    shuffle=False
)


# =====================================================================
# 4. CLASS WEIGHTS
# =====================================================================

final_bin_weights = (
    bin_weights
    .clone()
    .detach()
    .to(device)
)

final_multi_weights = (
    multi_weights
    .clone()
    .detach()
    .to(device)
)


# =====================================================================
# 5. REPRODUCIBILITY FUNCTION
# =====================================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =====================================================================
# 6. MEMORY WARM-UP
# =====================================================================

def warmup_memory(
    model,
    memory,
    graphs,
    batch_size=16
):

    memory.reset()

    loader = DataLoader(
        graphs,
        batch_size=batch_size,
        shuffle=False
    )

    model.eval()

    with torch.no_grad():

        for batch_idx, data in enumerate(loader):

            data = data.to(device)

            (
                _,
                _,
                _,
                node_emb,
                nids
            ) = model(
                data,
                memory
            )

            if nids is not None:

                memory.update_with_gru(
                    nids,
                    node_emb.detach(),
                    model.memory_updater,
                    device
                )

            if (
                batch_idx + 1
            ) % 1000 == 0:

                print(
                    f"  Warm-up batch "
                    f"{batch_idx+1:,} | "
                    f"memory={len(memory):,}"
                )


# =====================================================================
# 7. RESULT STORAGE
# =====================================================================

all_seed_results = []
all_histories = {}


# =====================================================================
# 8. FIVE INDEPENDENT RUNS
# =====================================================================

for seed_idx, seed in enumerate(
    SEEDS,
    start=1
):

    print("\n")
    print("=" * 100)

    print(
        f"SEED {seed} "
        f"({seed_idx}/{len(SEEDS)})"
    )

    print("=" * 100)


    # ================================================================
    # A. SET RANDOM SEED
    # ================================================================

    set_seed(seed)


    # ================================================================
    # B. FRESH MODEL
    # ================================================================

    model = TGNJointIDS(

        in_dim=IN_DIM,

        hidden_dim=HIDDEN_DIM,

        num_classes=NUM_CLASSES,

        memory_dim=MEMORY_DIM,

        num_layers=NUM_LAYERS,

        num_heads=NUM_HEADS,

        dropout=DROPOUT,

        edge_dim=EDGE_DIM,

        sagpool_ratio=SAGPOOL_R

    ).to(device)


    # Optional check that initializations really differ
    first_weights = (
        next(model.parameters())
        .detach()
        .view(-1)[:5]
        .cpu()
        .numpy()
    )

    print(
        "Initial weights:",
        first_weights
    )


    # ================================================================
    # C. FRESH TGN MEMORY
    # ================================================================

    tgn_memory = TGNNodeMemory(
        memory_dim=MEMORY_DIM
    )


    # ================================================================
    # D. FRESH TRAINER
    #
    # IMPORTANT:
    # Do NOT pass the old replay_buffer.
    #
    # JointTrainer will create a fresh:
    # deque(maxlen=5000)
    #
    # for every seed.
    #
    # Early stopping patience=20 is already inside trainer.train().
    # ================================================================

    trainer = JointTrainer(

        model=model,

        tgn_memory=tgn_memory,

        device=device,

        bin_weights=final_bin_weights,

        multi_weights=final_multi_weights,

        lr=1e-3,

        ewc_lambda=150,

        min_lr=1e-5,

        update_interval=50,

        replay_batch=64
    )


    # ================================================================
    # E. PHASE 1 — INITIAL TRAINING
    # ================================================================

    print("\n")
    print("-" * 90)
    print("PHASE 1 — INITIAL TRAINING")
    print("-" * 90)


    history = trainer.train(

        train_loader,

        val_loader,

        epochs=MAX_EPOCHS
    )


    all_histories[seed] = history


    n_epochs = len(
        history["train_loss"]
    )


    print("\nTraining completed.")

    print(
        f"Best validation accuracy: "
        f"{trainer.best_acc:.4f}"
    )

    print(
        f"Epochs executed: "
        f"{n_epochs}/{MAX_EPOCHS}"
    )


    # ================================================================
    # F. SAVE TRAINED MODEL IMMEDIATELY
    #
    # This is the IMPORTANT correction that prevents losing 4 hours.
    # ================================================================

    checkpoint_file = (
        f"TGN_JointIDS_seed_{seed}.pt"
    )


    torch.save(
        {

            "seed":
                seed,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                trainer.optimizer.state_dict(),

            "history":
                history,

            "best_val_acc":
                trainer.best_acc,

            "epochs_executed":
                n_epochs

        },

        checkpoint_file
    )


    print(
        f"✓ Trained checkpoint saved: "
        f"{checkpoint_file}"
    )


    # ================================================================
    # G. PHASE 2 — RESET + MEMORY WARM-UP
    # ================================================================

    print("\n")
    print("-" * 90)
    print("PHASE 2 — MEMORY WARM-UP")
    print("-" * 90)


    warmup_memory(

        model=model,

        memory=tgn_memory,

        graphs=train_graphs,

        batch_size=WARMUP_BATCH
    )


    warmup_memory_keys = len(
        tgn_memory
    )


    print(
        f"✓ Warm-up completed | "
        f"memory keys = "
        f"{warmup_memory_keys:,}"
    )


    # ================================================================
    # H. EWC REFERENCE
    # ================================================================

    print("\nComputing EWC reference...")


    ewc_loader = DataLoader(

        train_graphs[
            :EWC_N_GRAPHS
        ],

        batch_size=EWC_BATCH,

        shuffle=False
    )


    trainer.ewc = EWC(

        model,

        ewc_loader,

        device,

        lambda_ewc=150
    )


    print("✓ EWC reference created.")


    # ================================================================
    # I. PHASE 3 — STREAMING EVALUATION
    # ================================================================

    print("\n")
    print("-" * 90)
    print("PHASE 3 — STREAMING EVALUATION")
    print("-" * 90)


    y_bin_true = []
    y_bin_pred = []

    y_multi_true = []
    y_multi_pred = []

    latencies = []

    accumulated = []

    model.eval()


    for i, graph in enumerate(
        stream_graphs
    ):


        # ------------------------------------------------------------
        # Inference
        # ------------------------------------------------------------

        t0 = time.perf_counter()


        with torch.no_grad():

            batch = Batch.from_data_list(
                [graph]
            ).to(device)


            (
                bin_logits,
                multi_logits,
                _,
                node_emb,
                nids

            ) = model(

                batch,
                tgn_memory
            )


            bin_pred = int(

                bin_logits
                .argmax(dim=1)
                .item()
            )


            multi_pred = int(

                multi_logits
                .argmax(dim=1)
                .item()
            )


        # ------------------------------------------------------------
        # Temporal memory update
        # ------------------------------------------------------------

        if nids is not None:

            tgn_memory.update_with_gru(

                nids,

                node_emb.detach(),

                model.memory_updater,

                device
            )


        latency_ms = (

            time.perf_counter()
            - t0

        ) * 1000.0


        latencies.append(
            latency_ms
        )


        # ------------------------------------------------------------
        # Ground truth
        # ------------------------------------------------------------

        y_bin_true.append(

            int(
                graph.y_bin.item()
            )
        )


        y_bin_pred.append(
            bin_pred
        )


        y_multi_true.append(

            int(
                graph.y_multi.item()
            )
        )


        y_multi_pred.append(
            multi_pred
        )


        # ------------------------------------------------------------
        # Online adaptation every K = 32 graphs
        # ------------------------------------------------------------

        accumulated.append(
            graph
        )


        if len(
            accumulated
        ) >= UPDATE_INTERVAL:


            trainer.incremental_update(

                accumulated,

                n_steps=1
            )


            model.eval()

            accumulated = []


        # ------------------------------------------------------------
        # Progress
        # ------------------------------------------------------------

        if (
            i + 1
        ) % 5000 == 0:

            print(

                f"  {i+1:,}/"
                f"{len(stream_graphs):,} | "

                f"memory="
                f"{len(tgn_memory):,} | "

                f"mean latency="
                f"{np.mean(latencies):.3f} ms"
            )


    print(
        f"\n✓ Streaming completed: "
        f"{len(stream_graphs):,} windows"
    )


    # ================================================================
    # J. CONVERT RESULTS TO NUMPY
    # ================================================================

    y_bin_true = np.asarray(
        y_bin_true
    )

    y_bin_pred = np.asarray(
        y_bin_pred
    )

    y_multi_true = np.asarray(
        y_multi_true
    )

    y_multi_pred = np.asarray(
        y_multi_pred
    )


    # ================================================================
    # K. BINARY METRICS
    # ================================================================

    streaming_accuracy = accuracy_score(

        y_bin_true,

        y_bin_pred
    )


    # Attack-class F1
    binary_f1 = f1_score(

        y_bin_true,

        y_bin_pred,

        pos_label=1,

        zero_division=0
    )


    # Weighted binary F1
    binary_weighted_f1 = f1_score(

        y_bin_true,

        y_bin_pred,

        average="weighted",

        zero_division=0
    )


    tn, fp, fn, tp = confusion_matrix(

        y_bin_true,

        y_bin_pred,

        labels=[0, 1]

    ).ravel()


    benign_fpr = (

        fp / (fp + tn)

        if (fp + tn) > 0

        else np.nan
    )


    fnr = (

        fn / (fn + tp)

        if (fn + tp) > 0

        else np.nan
    )


    # ================================================================
    # L. ATTACK-ONLY MULTICLASS METRICS
    # ================================================================

    attack_mask = (
        y_bin_true == 1
    )


    y_true_attack = y_multi_true[
        attack_mask
    ]

    y_pred_attack = y_multi_pred[
        attack_mask
    ]


    attack_ids = [

        attack2id[
            attack
        ]

        for attack in ATTACK_CLASSES

        if attack != "benign"
    ]


    multiclass_weighted_f1 = f1_score(

        y_true_attack,

        y_pred_attack,

        labels=attack_ids,

        average="weighted",

        zero_division=0
    )


    multiclass_macro_f1 = f1_score(

        y_true_attack,

        y_pred_attack,

        labels=attack_ids,

        average="macro",

        zero_division=0
    )


    # ================================================================
    # M. PER-CLASS RECALL FUNCTION
    # ================================================================

    def get_recall(
        class_name
    ):

        cid = attack2id[
            class_name
        ]

        mask = (
            y_true_attack == cid
        )

        if mask.sum() == 0:

            return np.nan


        return float(

            np.mean(

                y_pred_attack[
                    mask
                ] == cid
            )
        )


    mitm_recall = get_recall(
        "mitm"
    )

    ransomware_recall = get_recall(
        "ransomware"
    )

    scanning_recall = get_recall(
        "scanning"
    )


    # ================================================================
    # N. RESULT FOR THIS SEED
    # ================================================================

    seed_result = {

        "seed":
            seed,

        "best_val_accuracy":
            trainer.best_acc,

        "epochs_executed":
            n_epochs,

        "stream_windows":
            len(stream_graphs),

        "streaming_accuracy":
            streaming_accuracy,

        "binary_f1":
            binary_f1,

        "binary_weighted_f1":
            binary_weighted_f1,

        "fnr":
            fnr,

        "benign_fpr":
            benign_fpr,

        "multiclass_weighted_f1":
            multiclass_weighted_f1,

        "multiclass_macro_f1":
            multiclass_macro_f1,

        "mitm_recall":
            mitm_recall,

        "ransomware_recall":
            ransomware_recall,

        "scanning_recall":
            scanning_recall,

        "mean_latency_ms":
            float(
                np.mean(latencies)
            ),

        "p95_latency_ms":
            float(
                np.percentile(
                    latencies,
                    95
                )
            ),

        "memory_keys":
            len(
                tgn_memory
            )
    }


    # ================================================================
    # O. APPEND RESULT
    #
    # THIS WAS MISSING FROM YOUR PREVIOUS RUN.
    # ================================================================

    all_seed_results.append(
        seed_result
    )


    # ================================================================
    # P. SAVE RESULT IMMEDIATELY
    # ================================================================

    pd.DataFrame(
        all_seed_results
    ).to_csv(

        "five_seed_results_partial.csv",

        index=False
    )


    print("\n")
    print("-" * 90)
    print(
        f"SEED {seed} FINAL RESULT"
    )
    print("-" * 90)


    for key, value in seed_result.items():

        if isinstance(
            value,
            (float, np.floating)
        ):

            print(
                f"{key:<32}: "
                f"{value:.4f}"
            )

        else:

            print(
                f"{key:<32}: "
                f"{value}"
            )


    # ================================================================
    # Q. OPTIONAL FINAL POST-STREAM CHECKPOINT
    # ================================================================

    torch.save(
        {

            "seed":
                seed,

            "model_state_dict":
                model.state_dict(),

            "stream_result":
                seed_result

        },

        f"TGN_JointIDS_seed_{seed}_poststream.pt"
    )


    print(
        f"\n✓ Seed {seed} completely saved."
    )


    # ================================================================
    # R. MEMORY CLEANUP
    # ================================================================

    del model
    del tgn_memory
    del trainer

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# =====================================================================
# 9. FINAL FIVE-SEED TABLE
# =====================================================================

df_seeds = pd.DataFrame(
    all_seed_results
)


print("\n")
print("=" * 120)
print("INDIVIDUAL SEED RESULTS")
print("=" * 120)

print(
    df_seeds.to_string(
        index=False
    )
)


# =====================================================================
# 10. MEAN ± STD
# =====================================================================

metric_cols = [

    "streaming_accuracy",

    "binary_f1",

    "binary_weighted_f1",

    "fnr",

    "benign_fpr",

    "multiclass_weighted_f1",

    "multiclass_macro_f1",

    "mitm_recall",

    "ransomware_recall",

    "scanning_recall"
]


summary = pd.DataFrame({

    "mean":
        df_seeds[
            metric_cols
        ].mean(),

    "std":
        df_seeds[
            metric_cols
        ].std(
            ddof=1
        )
})


print("\n")
print("=" * 90)
print("5-SEED MEAN ± STANDARD DEVIATION")
print("=" * 90)

print(
    summary.round(4)
)


# =====================================================================
# 11. PAPER-FRIENDLY FORMAT
# =====================================================================

paper_summary = []

for metric in metric_cols:

    mean_value = (
        summary.loc[
            metric,
            "mean"
        ]
    )

    std_value = (
        summary.loc[
            metric,
            "std"
        ]
    )

    paper_summary.append({

        "Metric":
            metric,

        "Mean":
            mean_value,

        "Std":
            std_value,

        "Mean ± Std":
            f"{mean_value:.4f} ± "
            f"{std_value:.4f}"
    })


paper_summary = pd.DataFrame(
    paper_summary
)


print("\n")
print("=" * 90)
print("PAPER TABLE")
print("=" * 90)

print(
    paper_summary.to_string(
        index=False
    )
)


# =====================================================================
# 12. SAVE FINAL RESULTS
# =====================================================================

df_seeds.to_csv(
    "five_seed_individual_results.csv",
    index=False
)

summary.to_csv(
    "five_seed_mean_std.csv"
)

paper_summary.to_csv(
    "five_seed_paper_table.csv",
    index=False
)


print("\n")
print("=" * 90)
print("ALL 5 SEEDS COMPLETED SUCCESSFULLY")
print("=" * 90)

print(
    "\nSaved:"
    "\n  five_seed_individual_results.csv"
    "\n  five_seed_mean_std.csv"
    "\n  five_seed_paper_table.csv"
    "\n  plus one trained checkpoint per seed."
)

In [ ]:
# =====================================================================
# OPERATING-POINT ANALYSIS (Section 5.6)
# Window composition + threshold sensitivity + Precision-Recall curve
#
# Uses:
#   - fixed train_graphs / val_graphs / stream_graphs
#   - trained seed-2026 checkpoint
#   - original TGN memory warm-up
#
# IMPORTANT:
# This cell performs inference only.
# =====================================================================

import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix
)


# =====================================================================
# 1. CONFIGURATION
# =====================================================================

SEED = 2026
CHECKPOINT = "TGN_JointIDS_seed_2026.pt"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 100)
print("OPERATING-POINT ANALYSIS (Section 5.6)")
print("WINDOW COMPOSITION + THRESHOLD SENSITIVITY + PR CURVE")
print("=" * 100)

print(f"Device     : {device}")
print(f"Checkpoint : {CHECKPOINT}")


# =====================================================================
# 2. WINDOW-LEVEL CLASS COMPOSITION
# =====================================================================

def class_composition(graphs, name):

    y = np.array([
        int(g.y_bin.item())
        for g in graphs
    ])

    n = len(y)

    benign = int((y == 0).sum())
    attack = int((y == 1).sum())

    return {
        "Split": name,
        "Total": n,
        "Benign": benign,
        "Benign (%)": 100 * benign / n,
        "Attack": attack,
        "Attack (%)": 100 * attack / n
    }


composition = pd.DataFrame([

    class_composition(
        train_graphs,
        "Train"
    ),

    class_composition(
        val_graphs,
        "Validation"
    ),

    class_composition(
        stream_graphs,
        "Streaming"
    )
])


print("\n")
print("=" * 100)
print("WINDOW-LEVEL CLASS COMPOSITION")
print("=" * 100)

print(
    composition.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

composition.to_csv(
    "window_class_composition.csv",
    index=False
)


# =====================================================================
# 3. RECREATE MODEL
# =====================================================================

model = TGNJointIDS(

    in_dim=IN_DIM,

    hidden_dim=HIDDEN_DIM,

    num_classes=NUM_CLASSES,

    memory_dim=MEMORY_DIM,

    num_layers=NUM_LAYERS,

    num_heads=NUM_HEADS,

    dropout=DROPOUT,

    edge_dim=EDGE_DIM,

    sagpool_ratio=SAGPOOL_R

).to(device)


# =====================================================================
# 4. LOAD SEED-2026 TRAINED CHECKPOINT
# =====================================================================

checkpoint = torch.load(
    CHECKPOINT,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    "\n✓ Seed-2026 trained model restored."
)


# =====================================================================
# 5. CREATE FRESH MEMORY
# =====================================================================

tgn_memory = TGNNodeMemory(
    memory_dim=MEMORY_DIM
)


# =====================================================================
# 6. MEMORY WARM-UP USING TRAINING SET
# =====================================================================

print("\nWarming up temporal memory...")


tgn_memory.reset()

warmup_loader = DataLoader(

    train_graphs,

    batch_size=16,

    shuffle=False
)


with torch.no_grad():

    for batch_idx, data in enumerate(
        warmup_loader
    ):

        data = data.to(device)

        (
            _,
            _,
            _,
            node_emb,
            nids

        ) = model(
            data,
            tgn_memory
        )

        if nids is not None:

            tgn_memory.update_with_gru(

                nids,

                node_emb.detach(),

                model.memory_updater,

                device
            )


        if (batch_idx + 1) % 1000 == 0:

            print(
                f"  Warm-up {batch_idx+1:,} batches | "
                f"memory={len(tgn_memory):,}"
            )


print(
    f"✓ Warm-up completed | "
    f"memory keys={len(tgn_memory):,}"
)


# =====================================================================
# 7. COLLECT BINARY PROBABILITIES ON STREAM
#
# IMPORTANT:
# We store P(attack), not just argmax predictions.
# These probabilities are needed for threshold analysis and PR curve.
# =====================================================================

print("\nCollecting streaming probabilities...")


y_true = []
p_attack = []

model.eval()


with torch.no_grad():

    for i, graph in enumerate(
        stream_graphs
    ):

        batch = Batch.from_data_list(
            [graph]
        ).to(device)


        (
            bin_logits,
            _,
            _,
            node_emb,
            nids

        ) = model(
            batch,
            tgn_memory
        )


        probabilities = F.softmax(
            bin_logits,
            dim=1
        )


        attack_probability = float(
            probabilities[0, 1].item()
        )


        y_true.append(
            int(graph.y_bin.item())
        )

        p_attack.append(
            attack_probability
        )


        # -------------------------------------------------------------
        # Update TGN memory exactly after observing this graph
        # -------------------------------------------------------------

        if nids is not None:

            tgn_memory.update_with_gru(

                nids,

                node_emb.detach(),

                model.memory_updater,

                device
            )


        if (i + 1) % 5000 == 0:

            print(
                f"  {i+1:,}/{len(stream_graphs):,}"
            )


y_true = np.asarray(
    y_true
)

p_attack = np.asarray(
    p_attack
)


print(
    f"\n✓ Probabilities collected for "
    f"{len(y_true):,} streaming windows."
)


# =====================================================================
# 8. THRESHOLD METRICS
# =====================================================================

def threshold_metrics(
    y_true,
    p_attack,
    threshold
):

    y_pred = (
        p_attack >= threshold
    ).astype(int)


    precision = precision_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )


    recall = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )


    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )


    tn, fp, fn, tp = confusion_matrix(

        y_true,
        y_pred,

        labels=[0, 1]

    ).ravel()


    fnr = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )


    benign_fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )


    return {

        "threshold": threshold,

        "precision": precision,

        "recall": recall,

        "f1": f1,

        "fnr": fnr,

        "benign_fpr": benign_fpr,

        "tn": tn,

        "fp": fp,

        "fn": fn,

        "tp": tp
    }


# =====================================================================
# 9. THRESHOLD SWEEP
# =====================================================================

thresholds = [

    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95
]


calibration_results = [

    threshold_metrics(
        y_true,
        p_attack,
        threshold
    )

    for threshold in thresholds
]


df_calib = pd.DataFrame(
    calibration_results
)


# =====================================================================
# 10. PRINT THRESHOLD TABLE
# =====================================================================

print("\n")
print("=" * 100)
print("BINARY THRESHOLD SENSITIVITY")
print("=" * 100)


display_table = df_calib[[
    "threshold",
    "precision",
    "recall",
    "f1",
    "fnr",
    "benign_fpr"
]].copy()


# Convert FNR/FPR to percentages
display_table["FNR (%)"] = (
    display_table["fnr"] * 100
)

display_table["Benign FPR (%)"] = (
    display_table["benign_fpr"] * 100
)


display_table = display_table.drop(
    columns=[
        "fnr",
        "benign_fpr"
    ]
)


print(
    display_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


df_calib.to_csv(
    "threshold_sensitivity_seed2026.csv",
    index=False
)


# =====================================================================
# 11. CHECK THRESHOLD = 0.50
# =====================================================================

baseline = df_calib[
    np.isclose(
        df_calib["threshold"],
        0.50
    )
].iloc[0]


print("\n")
print("=" * 100)
print("THRESHOLD = 0.50 CHECK")
print("=" * 100)

print(
    f"Accuracy    : "
    f"{(baseline['tp'] + baseline['tn']) / len(y_true):.4f}"
)

print(
    f"Precision   : "
    f"{baseline['precision']:.4f}"
)

print(
    f"Recall      : "
    f"{baseline['recall']:.4f}"
)

print(
    f"F1          : "
    f"{baseline['f1']:.4f}"
)

print(
    f"FNR         : "
    f"{baseline['fnr']*100:.2f}%"
)

print(
    f"Benign FPR  : "
    f"{baseline['benign_fpr']*100:.2f}%"
)

print(
    f"TN={int(baseline['tn'])}, "
    f"FP={int(baseline['fp'])}, "
    f"FN={int(baseline['fn'])}, "
    f"TP={int(baseline['tp'])}"
)


In [ ]:


# =====================================================================
# 12. PRECISION-RECALL CURVE
# =====================================================================

precision_curve, recall_curve, pr_thresholds = (
    precision_recall_curve(
        y_true,
        p_attack
    )
)


average_precision = (
    average_precision_score(
        y_true,
        p_attack
    )
)


print("\n")
print("=" * 100)
print("PRECISION-RECALL ANALYSIS")
print("=" * 100)

print(
    f"Average Precision (AP): "
    f"{average_precision:.6f}"
)


# =====================================================================
# 13. FIGURE
# =====================================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10, 3.2),
    dpi=200
)


# ---------------------------------------------------------------------
# (a) Precision-Recall curve
# ---------------------------------------------------------------------

ax = axes[0]

ax.plot(
    recall_curve,
    precision_curve,
    linewidth=1.8
)

ax.set_xlabel(
    "Recall"
)

ax.set_ylabel(
    "Precision"
)

ax.set_title(
    f"(a) Precision–Recall Curve\n"
    f"AP = {average_precision:.4f}"
)

ax.grid(
    alpha=0.25
)


# ---------------------------------------------------------------------
# (b) Precision / Recall / F1 vs threshold
# ---------------------------------------------------------------------

ax = axes[1]

ax.plot(
    df_calib["threshold"],
    df_calib["precision"],
    marker="o",
    label="Precision"
)

ax.plot(
    df_calib["threshold"],
    df_calib["recall"],
    marker="s",
    label="Recall"
)

ax.plot(
    df_calib["threshold"],
    df_calib["f1"],
    marker="^",
    label="F1"
)

ax.set_xlabel(
    "Attack Probability Threshold"
)

ax.set_ylabel(
    "Score"
)

ax.set_title(
    "(b) Threshold Sensitivity"
)

ax.legend(
    frameon=False,
    fontsize=8
)

ax.grid(
    alpha=0.25
)


# ---------------------------------------------------------------------
# (c) FPR / FNR vs threshold
# ---------------------------------------------------------------------

ax = axes[2]

ax.plot(
    df_calib["threshold"],
    df_calib["benign_fpr"] * 100,
    marker="o",
    label="Benign FPR"
)

ax.plot(
    df_calib["threshold"],
    df_calib["fnr"] * 100,
    marker="s",
    label="Attack FNR"
)

ax.set_xlabel(
    "Attack Probability Threshold"
)

ax.set_ylabel(
    "Rate (%)"
)

ax.set_title(
    "(c) FPR–FNR Trade-off"
)

ax.legend(
    frameon=False,
    fontsize=8
)

ax.grid(
    alpha=0.25
)


plt.tight_layout()


plt.savefig(
    "binary_operating_point_analysis_seed2026.pdf",
    bbox_inches="tight"
)

plt.savefig(
    "binary_operating_point_analysis_seed2026.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================
# 14. SAVE PR CURVE DATA
# =====================================================================

pd.DataFrame({

    "recall": recall_curve,

    "precision": precision_curve

}).to_csv(

    "precision_recall_curve.csv",

    index=False
)